# Trabajo Deep Learning 2026

**Clasificación multiclase de lesiones cutáneas con DermaMNIST**

En este notebook se implementa la comparación experimental del trabajo. Partimos de DermaMNIST y evaluamos un modelo entrenado desde cero, dos variantes de ResNet50 preentrenadas en ImageNet, dos variantes con un backbone médico RadImageNet/RAC y un modelo dermatológico EfficientNet-B1 preentrenado en el entorno ISIC.

Los resultados se comparan con métricas adaptadas al desbalanceo, repeticiones con distintas semillas y análisis visual mediante Grad-CAM.


## 1. Imports y configuración

En esta primera parte cargamos las librerías necesarias, definimos las rutas de trabajo en Drive y fijamos los parámetros generales del experimento.


In [ ]:
!pip -q install medmnist timm huggingface_hub gdown seaborn openpyxl opencv-python

In [ ]:
# ============================================================
# CACHÉ ROBUSTA EN DRIVE: DermaMNIST + pesos RAC/ISIC + progreso
# ============================================================

import os
import glob
import json
import time
import shutil
import zipfile
import tarfile
import subprocess
from pathlib import Path

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------
# 0. Asegurar dependencias
# ------------------------------------------------------------

def pip_install_if_needed(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except Exception:
        print(f"Instalando {package_name}...")
        subprocess.check_call(["python", "-m", "pip", "install", "-q", package_name])

pip_install_if_needed("medmnist", "medmnist")
pip_install_if_needed("gdown", "gdown")
pip_install_if_needed("huggingface_hub", "huggingface_hub")

from medmnist import DermaMNIST, INFO
import gdown
from huggingface_hub import hf_hub_download

# ------------------------------------------------------------
# 1. Asegurar rutas principales
# ------------------------------------------------------------

if "DRIVE_PROJECT_ROOT" not in globals():
    DRIVE_PROJECT_ROOT = "DRIVE_PROJECT_ROOT"

if "IMG_SIZE" not in globals():
    IMG_SIZE = 224

if "NUM_CLASSES" not in globals():
    NUM_CLASSES = 7

DATA_DIR = os.path.join(DRIVE_PROJECT_ROOT, "data", "dermamnist_224")
WEIGHTS_RAD_DIR = os.path.join(DRIVE_PROJECT_ROOT, "weights", "radimagenet_rac")
WEIGHTS_DERM_DIR = os.path.join(DRIVE_PROJECT_ROOT, "weights", "dermatology_maxnet_isic")
RESULTS_METRICS_DIR = os.path.join(DRIVE_PROJECT_ROOT, "results", "metrics")
RESULTS_PRED_DIR = os.path.join(DRIVE_PROJECT_ROOT, "results", "predictions")
REPORTS_DIR = os.path.join(DRIVE_PROJECT_ROOT, "reports")
LOGS_DIR = os.path.join(DRIVE_PROJECT_ROOT, "logs")

for d in [
    DATA_DIR,
    WEIGHTS_RAD_DIR,
    WEIGHTS_DERM_DIR,
    RESULTS_METRICS_DIR,
    RESULTS_PRED_DIR,
    REPORTS_DIR,
    LOGS_DIR,
]:
    os.makedirs(d, exist_ok=True)

RADIMAGENET_RESNET50_WEIGHTS_PATH = os.path.join(
    WEIGHTS_RAD_DIR,
    "resnet50_torch.pt"
)

MAXNET_ISIC_WEIGHTS_PATH = os.path.join(
    WEIGHTS_DERM_DIR,
    "best_weights.pth"
)

DERMAMNIST_CACHE_NPZ = os.path.join(
    DATA_DIR,
    f"dermamnist_{IMG_SIZE}_rgb_arrays_cached.npz"
)

print("DRIVE_PROJECT_ROOT:")
print(DRIVE_PROJECT_ROOT)
print("\nDATA_DIR:")
print(DATA_DIR)
print("\nDERMAMNIST_CACHE_NPZ:")
print(DERMAMNIST_CACHE_NPZ)

# ------------------------------------------------------------
# 2. Nombres de clases
# ------------------------------------------------------------

info = INFO["dermamnist"]
class_names = [info["label"][str(i)] for i in range(NUM_CLASSES)]

# ------------------------------------------------------------
# 3. Descargar/cargar DermaMNIST con caché propia en Drive
# ------------------------------------------------------------

def load_dermamnist_cached(size: int = 224, as_rgb: bool = True, use_cache: bool = True):
    """
    Carga DermaMNIST. Si existe un .npz propio en Drive, lo usa directamente.
    Si no existe, descarga/carga con medmnist y después guarda el .npz.
    """
    notes = []

    if use_cache and os.path.exists(DERMAMNIST_CACHE_NPZ):
        print("\nCargando DermaMNIST desde caché propia en Drive:")
        print(DERMAMNIST_CACHE_NPZ)

        data = np.load(DERMAMNIST_CACHE_NPZ, allow_pickle=True)

        bundle = {
            "X_train": data["X_train"],
            "X_val": data["X_val"],
            "X_test": data["X_test"],
            "y_train": data["y_train"],
            "y_val": data["y_val"],
            "y_test": data["y_test"],
            "class_names": list(data["class_names"]),
            "actual_size": int(data["actual_size"]),
            "notes": list(data["notes"]),
        }

        print("Caché cargada correctamente.")
        return bundle

    print("\nNo existe caché propia. Cargando/descargando DermaMNIST con medmnist...")

    try:
        train_ds = DermaMNIST(split="train", size=size, download=True, as_rgb=as_rgb, root=DATA_DIR)
        val_ds = DermaMNIST(split="val", size=size, download=True, as_rgb=as_rgb, root=DATA_DIR)
        test_ds = DermaMNIST(split="test", size=size, download=True, as_rgb=as_rgb, root=DATA_DIR)
        loaded_size = size

    except Exception as exc:
        notes.append(f"Fallo con size={size}: {exc}")
        notes.append("Se cargó DermaMNIST a size=128 y se redimensionó a 224x224.")
        print("Fallo cargando size=224. Intentando size=128 y redimensionando...")

        loaded_size = 128
        train_ds = DermaMNIST(split="train", size=128, download=True, as_rgb=as_rgb, root=DATA_DIR)
        val_ds = DermaMNIST(split="val", size=128, download=True, as_rgb=as_rgb, root=DATA_DIR)
        test_ds = DermaMNIST(split="test", size=128, download=True, as_rgb=as_rgb, root=DATA_DIR)

    X_train = train_ds.imgs.astype("uint8")
    X_val = val_ds.imgs.astype("uint8")
    X_test = test_ds.imgs.astype("uint8")

    y_train = train_ds.labels.squeeze().astype("int64")
    y_val = val_ds.labels.squeeze().astype("int64")
    y_test = test_ds.labels.squeeze().astype("int64")

    if loaded_size != size:
        X_train = tf.image.resize(X_train, (size, size)).numpy().astype("uint8")
        X_val = tf.image.resize(X_val, (size, size)).numpy().astype("uint8")
        X_test = tf.image.resize(X_test, (size, size)).numpy().astype("uint8")

    print("\nGuardando caché propia comprimida en Drive...")
    np.savez_compressed(
        DERMAMNIST_CACHE_NPZ,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        class_names=np.array(class_names, dtype=object),
        actual_size=np.array(size),
        notes=np.array(notes, dtype=object),
    )

    print("DermaMNIST guardado en:")
    print(DERMAMNIST_CACHE_NPZ)

    bundle = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "class_names": class_names,
        "actual_size": size,
        "notes": notes,
    }

    return bundle


dataset_bundle = load_dermamnist_cached(size=IMG_SIZE, as_rgb=True, use_cache=True)

X_train_raw = dataset_bundle["X_train"]
X_val_raw = dataset_bundle["X_val"]
X_test_raw = dataset_bundle["X_test"]

y_train = dataset_bundle["y_train"]
y_val = dataset_bundle["y_val"]
y_test = dataset_bundle["y_test"]

print("\n================ DATASET ================")
print("Train:", X_train_raw.shape, y_train.shape)
print("Val:  ", X_val_raw.shape, y_val.shape)
print("Test: ", X_test_raw.shape, y_test.shape)
print("Clases:", class_names)

if dataset_bundle["notes"]:
    note_path = os.path.join(REPORTS_DIR, "methodology_notes.md")
    with open(note_path, "a", encoding="utf-8") as f:
        f.write("\n## Nota sobre carga de DermaMNIST\n")
        for line in dataset_bundle["notes"]:
            f.write(f"- {line}\n")
    print("Se añadió una nota metodológica por fallback de tamaño.")

# ------------------------------------------------------------
# 4. Descargar ISIC / EfficientNet-B1 si no existe
# ------------------------------------------------------------

def download_isic_weights_if_needed():
    if os.path.exists(MAXNET_ISIC_WEIGHTS_PATH):
        print("\nISIC ya existe:")
        print(MAXNET_ISIC_WEIGHTS_PATH)
        return True

    print("\nDescargando ISIC/EfficientNet-B1 desde Hugging Face...")

    repo_id = "conan17970/efficientnet-b1-skin-cancer-isic2019"
    filename = "best_weights.pth"

    try:
        downloaded_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=WEIGHTS_DERM_DIR,
            local_dir_use_symlinks=False,
        )

        if downloaded_path != MAXNET_ISIC_WEIGHTS_PATH:
            shutil.copy2(downloaded_path, MAXNET_ISIC_WEIGHTS_PATH)

        print("ISIC descargado correctamente:")
        print(MAXNET_ISIC_WEIGHTS_PATH)
        return True

    except Exception as exc:
        print("No se pudo descargar ISIC automáticamente.")
        print("Error:", repr(exc))
        print("M6 debería quedar como NOT_RUN si falta este archivo.")
        return False

# ------------------------------------------------------------
# 5. Descargar RAC / RadImageNet si no existe
# ------------------------------------------------------------

def extract_if_archive(path, extract_dir):
    lower = path.lower()

    if zipfile.is_zipfile(path) or lower.endswith(".zip"):
        print("Extrayendo ZIP RadImageNet...")
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(path, "r") as zf:
            zf.extractall(extract_dir)
        return extract_dir

    if lower.endswith((".tar", ".tar.gz", ".tgz")):
        print("Extrayendo TAR RadImageNet...")
        os.makedirs(extract_dir, exist_ok=True)
        with tarfile.open(path, "r:*") as tf:
            tf.extractall(extract_dir)
        return extract_dir

    return os.path.dirname(path)


def find_resnet50_weight_file(search_dir):
    patterns = [
        "**/*resnet*50*.pt",
        "**/*resnet*50*.pth",
        "**/*ResNet*50*.pt",
        "**/*ResNet*50*.pth",
        "**/*resnet50*.ckpt",
        "**/*ResNet50*.ckpt",
        "**/*resnet50*.h5",
        "**/*ResNet50*.h5",
    ]

    candidates = []
    for pat in patterns:
        candidates.extend(glob.glob(os.path.join(search_dir, pat), recursive=True))

    candidates = sorted(
        set(candidates),
        key=lambda p: (
            0 if p.lower().endswith((".pt", ".pth")) else 1,
            len(p),
        ),
    )

    return candidates[0] if candidates else None


def download_radimagenet_weights_if_needed():
    if os.path.exists(RADIMAGENET_RESNET50_WEIGHTS_PATH):
        print("\nRAC/RadImageNet ya existe:")
        print(RADIMAGENET_RESNET50_WEIGHTS_PATH)
        return True

    print("\nDescargando RadImageNet/RAC oficial desde project_storage...")

    # Enlace PyTorch oficial del repositorio BMEII-AI/RadImageNet.
    drive_file_id = "1RHt2GnuOYlc_gcoTETtBDSW73mFyRAtR"
    raw_download_path = os.path.join(WEIGHTS_RAD_DIR, "radimagenet_pytorch_official_download")

    try:
        url = f"https://drive.google.com/uc?id={drive_file_id}"

        downloaded_path = gdown.download(
            url=url,
            output=raw_download_path,
            quiet=False,
            fuzzy=True,
        )

        if downloaded_path is None or not os.path.exists(downloaded_path):
            raise RuntimeError("gdown no devolvió un archivo válido.")

        # Si es ZIP pero no tiene extensión, renombrar.
        if zipfile.is_zipfile(downloaded_path) and not downloaded_path.endswith(".zip"):
            zip_path = downloaded_path + ".zip"
            os.rename(downloaded_path, zip_path)
            downloaded_path = zip_path

        extract_dir = os.path.join(WEIGHTS_RAD_DIR, "radimagenet_pytorch_extracted")
        search_dir = extract_if_archive(downloaded_path, extract_dir)

        candidate = find_resnet50_weight_file(search_dir)

        if candidate is None and downloaded_path.lower().endswith((".pt", ".pth", ".ckpt")):
            candidate = downloaded_path

        if candidate is None:
            raise FileNotFoundError("No se encontró ningún archivo ResNet50 reconocible.")

        shutil.copy2(candidate, RADIMAGENET_RESNET50_WEIGHTS_PATH)

        print("RadImageNet/RAC preparado correctamente:")
        print("Origen:", candidate)
        print("Final: ", RADIMAGENET_RESNET50_WEIGHTS_PATH)
        return True

    except Exception as exc:
        print("No se pudo preparar RadImageNet/RAC automáticamente.")
        print("Error:", repr(exc))
        print("Si falta este archivo, M4/M5 deberían quedar como NOT_RUN:")
        print(RADIMAGENET_RESNET50_WEIGHTS_PATH)
        return False


ok_isic = download_isic_weights_if_needed()
ok_rac = download_radimagenet_weights_if_needed()

print("\n================ PESOS EXTERNOS ================")
print("ISIC disponible:", ok_isic, "|", os.path.exists(MAXNET_ISIC_WEIGHTS_PATH))
print("RAC disponible: ", ok_rac, "|", os.path.exists(RADIMAGENET_RESNET50_WEIGHTS_PATH))

print("\nContenido weights/radimagenet_rac:")
print(os.listdir(WEIGHTS_RAD_DIR))

print("\nContenido weights/dermatology_maxnet_isic:")
print(os.listdir(WEIGHTS_DERM_DIR))

# ------------------------------------------------------------
# 6. Comprobar qué modelos/semillas parecen ya completados
# ------------------------------------------------------------

def completed_run_exists(model_id, seed):
    metrics_path = os.path.join(RESULTS_METRICS_DIR, f"{model_id}_seed{seed}_metrics.json")
    pred_path = os.path.join(RESULTS_PRED_DIR, f"{model_id}_seed{seed}_predictions.csv")
    return os.path.exists(metrics_path) and os.path.exists(pred_path)


if "MODEL_DISPLAY_NAMES" in globals():
    print("\n================ PROGRESO DETECTADO ================")

    seeds_to_check = SEEDS if "SEEDS" in globals() else [123]

    for model_id, model_name in MODEL_DISPLAY_NAMES.items():
        states = []
        for seed in seeds_to_check:
            states.append(completed_run_exists(model_id, seed))

        if all(states):
            status = "COMPLETO"
        elif any(states):
            status = "PARCIAL"
        else:
            status = "PENDIENTE"

        print(f"{model_name}: {status}")

    print("\nSi Colab se corta:")
    print("1. Vuelve a ejecutar hasta esta celda.")
    print("2. Mira qué modelos aparecen como COMPLETO.")
    print("3. Pon RUN_Mx=False en los modelos completados para no repetirlos.")
    print("4. Deja RUN_Mx=True solo en los pendientes.")
else:
    print("\nMODEL_DISPLAY_NAMES aún no está definido. La comprobación de progreso se hará más adelante.")

In [ ]:
from google.colab import drive
drive.mount('project_storage')

Antes de construir este notebook se revisaron los notebooks previos de prácticas y del trabajo. Se reutilizó la estructura de la CNN propia, funciones de evaluación, curvas de entrenamiento, matrices de confusión y Grad-CAM cuando fue posible.


In [ ]:
AUDIT_INFO = {
    "summary": (
        "Antes de construir este notebook se revisaron las prácticas previas y las "
        "versiones intermedias del trabajo para recuperar las partes ya consolidadas."
    ),
    "reused": [
        "Arquitectura base de la CNN propia.",
        "Funciones de evaluación en test y cálculo de métricas.",
        "Curvas de entrenamiento y matrices de confusión.",
        "Implementación de Grad-CAM adaptada al mejor modelo justo.",
    ],
    "discarded": [
        "Pruebas parciales sin tabla final unificada.",
        "Experimentos auxiliares que no forman parte de la comparación principal de seis modelos.",
    ],
}

print(AUDIT_INFO["summary"])


### Rutas y guardado en Drive


In [ ]:
import os
import sys
import json
import time
import math
import random
import shutil
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as resnet50_preprocess

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50 as torch_resnet50
import timm
from huggingface_hub import hf_hub_download

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

from medmnist import DermaMNIST, INFO

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    import os

    MOUNT_POINT = "project_storage"

    if os.path.exists(os.path.join(MOUNT_POINT, "PROJECT_STORAGE")):
        print("project_storage ya está montado en:", MOUNT_POINT)
    else:
        print("project_storage no parece montado. Intentando montar...")
        try:
            drive.mount(MOUNT_POINT)
        except ValueError as e:
            print("Aviso al montar Drive:", e)
            print("Contenido actual de project_storage:")
            print(os.listdir(MOUNT_POINT) if os.path.exists(MOUNT_POINT) else "No existe")
            if os.path.exists(os.path.join(MOUNT_POINT, "PROJECT_STORAGE")):
                print("PROJECT_STORAGE existe; continuamos sin volver a montar.")
            else:
                raise
DRIVE_PROJECT_ROOT = "DRIVE_PROJECT_ROOT"

DATA_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "data")
DATA_DIR = os.path.join(DATA_ROOT, "dermamnist_224")
WEIGHTS_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "weights")
WEIGHTS_IMAGENET_DIR = os.path.join(WEIGHTS_ROOT, "imagenet")
WEIGHTS_RAD_DIR = os.path.join(WEIGHTS_ROOT, "radimagenet_rac")
WEIGHTS_DERM_DIR = os.path.join(WEIGHTS_ROOT, "dermatology_maxnet_isic")
MODELS_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "models")
CHECKPOINT_DIR = os.path.join(MODELS_ROOT, "checkpoints")
FINAL_MODEL_DIR = os.path.join(MODELS_ROOT, "final_models")
RESULTS_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "results")
HISTORIES_DIR = os.path.join(RESULTS_ROOT, "histories")
PREDICTIONS_DIR = os.path.join(RESULTS_ROOT, "predictions")
METRICS_DIR = os.path.join(RESULTS_ROOT, "metrics")
CLASS_REPORT_DIR = os.path.join(RESULTS_ROOT, "classification_reports")
SUMMARY_DIR = os.path.join(RESULTS_ROOT, "statistical_summary")
FIGURES_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "figures")
CLASS_DIST_DIR = os.path.join(FIGURES_ROOT, "class_distribution")
LEARNING_CURVES_DIR = os.path.join(FIGURES_ROOT, "learning_curves")
CM_DIR = os.path.join(FIGURES_ROOT, "confusion_matrices")
MODEL_COMPARISON_DIR = os.path.join(FIGURES_ROOT, "model_comparison")
EXAMPLES_DIR = os.path.join(FIGURES_ROOT, "correct_incorrect_examples")
GRADCAM_DIR = os.path.join(FIGURES_ROOT, "gradcam")
LOG_DIR = os.path.join(DRIVE_PROJECT_ROOT, "logs")
REPORTS_DIR = os.path.join(DRIVE_PROJECT_ROOT, "reports")

for folder in [
    DATA_ROOT,
    DATA_DIR,
    WEIGHTS_ROOT,
    WEIGHTS_IMAGENET_DIR,
    WEIGHTS_RAD_DIR,
    WEIGHTS_DERM_DIR,
    MODELS_ROOT,
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR,
    RESULTS_ROOT,
    HISTORIES_DIR,
    PREDICTIONS_DIR,
    METRICS_DIR,
    CLASS_REPORT_DIR,
    SUMMARY_DIR,
    FIGURES_ROOT,
    CLASS_DIST_DIR,
    LEARNING_CURVES_DIR,
    CM_DIR,
    MODEL_COMPARISON_DIR,
    EXAMPLES_DIR,
    GRADCAM_DIR,
    LOG_DIR,
    REPORTS_DIR,
]:
    os.makedirs(folder, exist_ok=True)

print("Proyecto en Drive:", DRIVE_PROJECT_ROOT)


### Parámetros del experimento


In [ ]:
PROJECT_NAME = "DermaMNIST-skin-lesion-classification"
IMG_SIZE = 224
NUM_CLASSES = 7
BATCH_SIZE = 16

EPOCHS_HEAD = 30
EPOCHS_DEEP = 30
LR_HEAD = 1e-4
LR_DEEP = 1e-5

SEEDS = [123]
N_REPEATS = len(SEEDS)

FAST_DEBUG = False

CLASS_WEIGHT_STRATEGY = "balanced_smoothed"
USE_SMOOTHED_CLASS_WEIGHTS_FOR_ALL = True

AUG_ROTATION_DEGREES = 10
AUG_ZOOM = 0.10
AUG_TRANSLATION = 0.05
AUG_CONTRAST = 0.10
AUG_HORIZONTAL_FLIP = True
AUG_TORCH_COLOR_JITTER = 0.05

EARLY_STOPPING_MONITOR = "val_macro_f1"
CHECKPOINT_MONITOR = "val_macro_f1"

MODEL_DISPLAY_NAMES = {
    "M1_CNN_PROPIA_224": "M1: CNN propia",
    "M2_RESNET50_IMAGENET_SHALLOW": "M2: ResNet50 ImageNet shallow",
    "M3_RESNET50_IMAGENET_DEEP": "M3: ResNet50 ImageNet deep",
    "M4_RESNET50_RADIMAGENET_RAC_SHALLOW": "M4: ResNet50 RadImageNet/RAC shallow",
    "M5_RESNET50_RADIMAGENET_RAC_DEEP": "M5: ResNet50 RadImageNet/RAC deep",
    "M6_MAXNET_ISIC_DERMATOLOGY_FROZEN_BASELINE": "M6: EfficientNet-B1 ISIC frozen",
}
MODEL_ORDER = list(MODEL_DISPLAY_NAMES.keys())

if FAST_DEBUG:
    SEEDS = [17]
    N_REPEATS = len(SEEDS)
    EPOCHS_HEAD = 3
    EPOCHS_DEEP = 3

EVALUATE_VAL_F1_EVERY_N_EPOCHS = 1

RUN_M1 = True
RUN_M2 = True
RUN_M3 = True

# Déjalos en False hasta tener los pesos RAC/RadImageNet subidos.
RUN_M4 = True
RUN_M5 = True

# Este puede quedarse True si quieres que intente descargar/cargar ISIC.
RUN_M6 = True

PALETTE = {
    "primary": "#0f766e",
    "secondary": "#0891b2",
    "accent": "#14b8a6",
    "light": "#ecfeff",
}

sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": "white",
    "grid.color": "#dbeafe",
    "figure.facecolor": "white",
})

print("FAST_DEBUG:", FAST_DEBUG)
print("Seeds:", SEEDS)
print("N_REPEATS:", N_REPEATS)
print("EPOCHS_HEAD:", EPOCHS_HEAD, "| EPOCHS_DEEP:", EPOCHS_DEEP)

### Reproducibilidad


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def log_error(message: str) -> None:
    with open(os.path.join(LOG_DIR, "run_errors.txt"), "a", encoding="utf-8") as f:
        f.write(f"[{datetime.now().isoformat()}] {message}\n")


def package_versions() -> dict:
    versions = {
        "python": sys.version.replace("\n", " "),
        "tensorflow": tf.__version__,
        "keras": getattr(keras, "__version__", "unknown"),
        "medmnist": __import__("medmnist").__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": __import__("sklearn").__version__,
        "matplotlib": plt.matplotlib.__version__,
        "opencv": cv2.__version__,
        "torch": torch.__version__,
        "torchvision": __import__("torchvision").__version__,
        "timm": timm.__version__,
    }
    return versions


def save_environment_versions() -> None:
    versions = package_versions()
    path = os.path.join(REPORTS_DIR, "environment_versions.txt")
    with open(path, "w", encoding="utf-8") as f:
        for key, value in versions.items():
            f.write(f"{key}: {value}\n")
    print("Environment versions guardadas en:", path)


save_environment_versions()
print("GPU disponibles TensorFlow:", tf.config.list_physical_devices("GPU"))
print("Device PyTorch:", device)

## 2. Carga de DermaMNIST

Utilizamos las particiones oficiales de entrenamiento, validación y test. El objetivo es mantener la comparación homogénea entre todos los modelos.


In [ ]:
# ============================================================
# MONTAJE LIMPIO DE GOOGLE DRIVE + RECUPERACIÓN DE CACHÉ/PESOS
# ============================================================

import os
import shutil
from google.colab import drive

# 1. Montar Drive en una ruta limpia para evitar el conflicto con project_storage
GDRIVE_MOUNT = "project_storage"

if os.path.exists(os.path.join(GDRIVE_MOUNT, "PROJECT_STORAGE")):
    print("project_storage ya está montado en:", GDRIVE_MOUNT)
else:
    print("Montando project_storage en:", GDRIVE_MOUNT)
    drive.mount(GDRIVE_MOUNT, force_remount=False)

# 2. Usar SIEMPRE esta ruta a partir de ahora
DRIVE_PROJECT_ROOT = os.path.join(
    GDRIVE_MOUNT,
    "PROJECT_STORAGE",
    "Master",
    "Segundo 4",
    "Deep Learning",
    "DermaMNIST-skin-lesion-classification"
)

DATA_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "data")
DATA_DIR = os.path.join(DATA_ROOT, "dermamnist_224")

WEIGHTS_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "weights")
WEIGHTS_RAD_DIR = os.path.join(WEIGHTS_ROOT, "radimagenet_rac")
WEIGHTS_DERM_DIR = os.path.join(WEIGHTS_ROOT, "dermatology_maxnet_isic")

RESULTS_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "results")
HISTORIES_DIR = os.path.join(RESULTS_ROOT, "histories")
PREDICTIONS_DIR = os.path.join(RESULTS_ROOT, "predictions")
METRICS_DIR = os.path.join(RESULTS_ROOT, "metrics")
CLASS_REPORT_DIR = os.path.join(RESULTS_ROOT, "classification_reports")
SUMMARY_DIR = os.path.join(RESULTS_ROOT, "statistical_summary")

MODELS_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "models")
CHECKPOINT_DIR = os.path.join(MODELS_ROOT, "checkpoints")
FINAL_MODEL_DIR = os.path.join(MODELS_ROOT, "final_models")

FIGURES_ROOT = os.path.join(DRIVE_PROJECT_ROOT, "figures")
CLASS_DIST_DIR = os.path.join(FIGURES_ROOT, "class_distribution")
LEARNING_CURVES_DIR = os.path.join(FIGURES_ROOT, "learning_curves")
CM_DIR = os.path.join(FIGURES_ROOT, "confusion_matrices")
MODEL_COMPARISON_DIR = os.path.join(FIGURES_ROOT, "model_comparison")
EXAMPLES_DIR = os.path.join(FIGURES_ROOT, "correct_incorrect_examples")
GRADCAM_DIR = os.path.join(FIGURES_ROOT, "gradcam")

LOG_DIR = os.path.join(DRIVE_PROJECT_ROOT, "logs")
REPORTS_DIR = os.path.join(DRIVE_PROJECT_ROOT, "reports")

for folder in [
    DATA_ROOT, DATA_DIR,
    WEIGHTS_ROOT, WEIGHTS_RAD_DIR, WEIGHTS_DERM_DIR,
    RESULTS_ROOT, HISTORIES_DIR, PREDICTIONS_DIR, METRICS_DIR,
    CLASS_REPORT_DIR, SUMMARY_DIR,
    MODELS_ROOT, CHECKPOINT_DIR, FINAL_MODEL_DIR,
    FIGURES_ROOT, CLASS_DIST_DIR, LEARNING_CURVES_DIR, CM_DIR,
    MODEL_COMPARISON_DIR, EXAMPLES_DIR, GRADCAM_DIR,
    LOG_DIR, REPORTS_DIR,
]:
    os.makedirs(folder, exist_ok=True)

# 3. Rutas esperadas
DERMAMNIST_CACHE_NPZ = os.path.join(
    DATA_DIR,
    f"dermamnist_{IMG_SIZE}_rgb_arrays_cached.npz"
)

RADIMAGENET_RESNET50_WEIGHTS_PATH = os.path.join(
    WEIGHTS_RAD_DIR,
    "resnet50_torch.pt"
)

MAXNET_ISIC_WEIGHTS_PATH = os.path.join(
    WEIGHTS_DERM_DIR,
    "best_weights.pth"
)

# 4. Intentar recuperar cosas que quizá se quedaron en project_storage local
OLD_ROOT = "DRIVE_PROJECT_ROOT"

old_cache = os.path.join(
    OLD_ROOT,
    "data",
    "dermamnist_224",
    f"dermamnist_{IMG_SIZE}_rgb_arrays_cached.npz"
)

old_rac = os.path.join(
    OLD_ROOT,
    "weights",
    "radimagenet_rac",
    "resnet50_torch.pt"
)

old_isic = os.path.join(
    OLD_ROOT,
    "weights",
    "dermatology_maxnet_isic",
    "best_weights.pth"
)

def copy_if_missing(old_path, new_path, label):
    if os.path.exists(new_path):
        print(f"{label} ya existe en Drive real:", new_path)
    elif os.path.exists(old_path):
        os.makedirs(os.path.dirname(new_path), exist_ok=True)
        print(f"Copiando {label} desde ruta antigua/local a Drive real...")
        shutil.copy2(old_path, new_path)
        print("Copiado a:", new_path)
    else:
        print(f"{label} no encontrado todavía.")

copy_if_missing(old_cache, DERMAMNIST_CACHE_NPZ, "Caché DermaMNIST")
copy_if_missing(old_rac, RADIMAGENET_RESNET50_WEIGHTS_PATH, "Peso RAC/RadImageNet")
copy_if_missing(old_isic, MAXNET_ISIC_WEIGHTS_PATH, "Peso ISIC")

print("\n================ COMPROBACIÓN FINAL ================")
print("DRIVE_PROJECT_ROOT:", DRIVE_PROJECT_ROOT)
print("Existe PROJECT_STORAGE real:", os.path.exists(os.path.join(GDRIVE_MOUNT, "PROJECT_STORAGE")))
print("Caché DermaMNIST:", os.path.exists(DERMAMNIST_CACHE_NPZ), DERMAMNIST_CACHE_NPZ)
print("Peso RAC:", os.path.exists(RADIMAGENET_RESNET50_WEIGHTS_PATH), RADIMAGENET_RESNET50_WEIGHTS_PATH)
print("Peso ISIC:", os.path.exists(MAXNET_ISIC_WEIGHTS_PATH), MAXNET_ISIC_WEIGHTS_PATH)

In [ ]:
# ============================================================
# 2. Carga de DermaMNIST desde caché propia en Drive
# ============================================================

import os
import numpy as np
import tensorflow as tf
from medmnist import INFO

# ------------------------------------------------------------
# Comprobar que estamos usando la ruta buena de project_storage
# ------------------------------------------------------------

if "DRIVE_PROJECT_ROOT" not in globals():
    DRIVE_PROJECT_ROOT = "PROJECT_STORAGE/DermaMNIST-skin-lesion-classification"

if "DATA_DIR" not in globals():
    DATA_DIR = os.path.join(DRIVE_PROJECT_ROOT, "data", "dermamnist_224")

if "REPORTS_DIR" not in globals():
    REPORTS_DIR = os.path.join(DRIVE_PROJECT_ROOT, "reports")
    os.makedirs(REPORTS_DIR, exist_ok=True)

if "IMG_SIZE" not in globals():
    IMG_SIZE = 224

if "NUM_CLASSES" not in globals():
    NUM_CLASSES = 7

info = INFO["dermamnist"]
class_names = [info["label"][str(i)] for i in range(NUM_CLASSES)]

DERMAMNIST_CACHE_NPZ = os.path.join(
    DATA_DIR,
    f"dermamnist_{IMG_SIZE}_rgb_arrays_cached.npz"
)

print("Ruta de proyecto:")
print(DRIVE_PROJECT_ROOT)

print("\nRuta de caché DermaMNIST:")
print(DERMAMNIST_CACHE_NPZ)

# ------------------------------------------------------------
# Cargar desde caché
# ------------------------------------------------------------

if not os.path.exists(DERMAMNIST_CACHE_NPZ):
    raise FileNotFoundError(
        "No existe la caché propia de DermaMNIST. "
        "No ejecutes descarga otra vez todavía. Revisa que DRIVE_PROJECT_ROOT apunte a project_storage."
    )

print("\nCargando DermaMNIST desde caché propia en Drive...")

data = np.load(DERMAMNIST_CACHE_NPZ, allow_pickle=True)

X_train_raw = data["X_train"]
X_val_raw = data["X_val"]
X_test_raw = data["X_test"]

y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

if "class_names" in data.files:
    class_names = list(data["class_names"])
else:
    class_names = [info["label"][str(i)] for i in range(NUM_CLASSES)]

notes = list(data["notes"]) if "notes" in data.files else []

print("\n================ DATASET ================")
print("Desde caché: True")
print("Train:", X_train_raw.shape, y_train.shape)
print("Val:  ", X_val_raw.shape, y_val.shape)
print("Test: ", X_test_raw.shape, y_test.shape)
print("Clases:", class_names)

# ------------------------------------------------------------
# Comprobaciones rápidas
# ------------------------------------------------------------

assert X_train_raw.shape[1:] == (IMG_SIZE, IMG_SIZE, 3), X_train_raw.shape
assert X_val_raw.shape[1:] == (IMG_SIZE, IMG_SIZE, 3), X_val_raw.shape
assert X_test_raw.shape[1:] == (IMG_SIZE, IMG_SIZE, 3), X_test_raw.shape

assert len(np.unique(y_train)) == NUM_CLASSES, np.unique(y_train)
assert len(class_names) == NUM_CLASSES, class_names

print("\nCarga correcta. Ya puedes ejecutar la exploración del dataset.")

## 3. Exploración del dataset

En esta sección comprobamos el tamaño de cada partición, la distribución por clases y algunos ejemplos de imágenes. Dado el desbalanceo de DermaMNIST, más adelante priorizaremos métricas macro y balanced accuracy.


In [ ]:
def class_distribution_dataframe(y, split_name: str) -> pd.DataFrame:
    counts = pd.Series(y).value_counts().sort_index()
    df = pd.DataFrame({
        "split": split_name,
        "class_id": counts.index,
        "class_name": [class_names[i] for i in counts.index],
        "count": counts.values,
    })
    df["percentage"] = 100 * df["count"] / df["count"].sum()
    return df


def plot_class_distribution(df: pd.DataFrame, split_name: str) -> None:
    plt.figure(figsize=(11, 4))
    sns.barplot(data=df, x="class_id", y="count", color=PALETTE["secondary"])
    plt.xticks(range(len(class_names)), class_names, rotation=35, ha="right")
    plt.title(f"Distribución de clases — {split_name}")
    plt.xlabel("Clase")
    plt.ylabel("Número de imágenes")
    plt.tight_layout()
    png_path = os.path.join(CLASS_DIST_DIR, f"{split_name.lower()}_class_distribution.png")
    pdf_path = os.path.join(CLASS_DIST_DIR, f"{split_name.lower()}_class_distribution.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.show()


def inspect_dataset(X: np.ndarray, y: np.ndarray, n: int = 8, title: str = "Ejemplos") -> None:
    rng = np.random.default_rng(17)
    idx = rng.choice(len(X), size=min(n, len(X)), replace=False)
    plt.figure(figsize=(14, 4))
    for j, i in enumerate(idx):
        plt.subplot(1, len(idx), j + 1)
        plt.imshow(X[i])
        plt.title(class_names[int(y[i])], fontsize=8)
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


train_dist = class_distribution_dataframe(y_train, "train")
val_dist = class_distribution_dataframe(y_val, "val")
test_dist = class_distribution_dataframe(y_test, "test")
class_dist_all = pd.concat([train_dist, val_dist, test_dist], ignore_index=True)
class_dist_all.to_csv(os.path.join(CLASS_DIST_DIR, "class_distribution_all_splits.csv"), index=False)

display(train_dist)
display(val_dist)
display(test_dist)
plot_class_distribution(train_dist, "train")
plot_class_distribution(val_dist, "val")
plot_class_distribution(test_dist, "test")
inspect_dataset(X_train_raw, y_train, n=8, title="Ejemplos de DermaMNIST train")
print("Advertencia: DermaMNIST está desbalanceado; la métrica principal será Macro F1.")

## 4. Preprocesado

Aplicamos un preprocesado coherente con cada familia de modelos. La augmentación se limita al conjunto de entrenamiento y se mantiene moderada para no alterar la semántica de la lesión.


In [ ]:
# ============================================================
# Pesos de clase, augmentation y datasets TensorFlow
# ============================================================

def compute_class_weights_dict(y: np.ndarray, smooth: bool = False) -> dict:
    """
    Calcula pesos de clase balanceados.

    Si smooth=True:
    - se aplica raíz cuadrada a los pesos balanceados;
    - se recortan entre 0.7 y 2.5;
    - esto evita pesos extremos en clases muy minoritarias.
    """
    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=y
    )

    if smooth:
        weights = np.clip(np.sqrt(weights), 0.7, 2.5)

    return {int(i): float(w) for i, w in enumerate(weights)}


# Estrategia final: pesos suavizados para TODOS los modelos
CLASS_WEIGHT_STRATEGY = "balanced_smoothed"
USE_SMOOTHED_CLASS_WEIGHTS_FOR_ALL = True

class_weights_full = compute_class_weights_dict(y_train, smooth=False)
class_weights_smooth = compute_class_weights_dict(y_train, smooth=True)

if USE_SMOOTHED_CLASS_WEIGHTS_FOR_ALL:
    class_weights_main = class_weights_smooth
else:
    class_weights_main = class_weights_full

print("CLASS_WEIGHT_STRATEGY:", CLASS_WEIGHT_STRATEGY)
print("USE_SMOOTHED_CLASS_WEIGHTS_FOR_ALL:", USE_SMOOTHED_CLASS_WEIGHTS_FOR_ALL)
print("class_weights_main:", class_weights_main)
print("class_weights_full:", class_weights_full)
print("class_weights_smooth:", class_weights_smooth)


def build_augmentation_layer():
    aug_layers = []

    if AUG_HORIZONTAL_FLIP:
        aug_layers.append(layers.RandomFlip("horizontal"))

    aug_layers.extend([
        layers.RandomRotation(AUG_ROTATION_DEGREES / 360.0),
        layers.RandomZoom(AUG_ZOOM),
        layers.RandomTranslation(AUG_TRANSLATION, AUG_TRANSLATION),
        layers.RandomContrast(AUG_CONTRAST),
    ])

    return keras.Sequential(aug_layers, name="augmentation")


def get_preprocess_function(model_type: str):
    mapping = {
        "cnn": lambda x: x / 255.0,
        "resnet50_imagenet": resnet50_preprocess,
        "radimagenet_rac": "torch_-1_1",
        "maxnet_isic": "torch_imagenet_norm",
    }
    return mapping[model_type]


def prepare_tf_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size: int = 32
):
    autotune = tf.data.AUTOTUNE

    train_ds = (
        tf.data.Dataset
        .from_tensor_slices((X_train, y_train))
        .shuffle(len(X_train), seed=17)
        .batch(batch_size)
        .prefetch(autotune)
    )

    val_ds = (
        tf.data.Dataset
        .from_tensor_slices((X_val, y_val))
        .batch(batch_size)
        .prefetch(autotune)
    )

    test_ds = (
        tf.data.Dataset
        .from_tensor_slices((X_test, y_test))
        .batch(batch_size)
        .prefetch(autotune)
    )

    return train_ds, val_ds, test_ds


train_tf_ds, val_tf_ds, test_tf_ds = prepare_tf_datasets(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    X_test_raw,
    y_test,
    batch_size=BATCH_SIZE
)

print("tf.data listo:", train_tf_ds, val_tf_ds, test_tf_ds)

## 5. Definición de métricas

La métrica principal del trabajo es `f1_macro`. También se recogen balanced accuracy, kappa, accuracy, informes por clase y matrices de confusión.


In [ ]:
ALL_RESULTS = []
MODEL_STATUSES = {}
BEST_RUN_CACHE = {}


def safe_name(text: str) -> str:
    keep = "".join(ch if ch.isalnum() or ch in "_-" else "_" for ch in text.lower())
    while "__" in keep:
        keep = keep.replace("__", "_")
    return keep.strip("_")


class TrainValMacroF1Callback(Callback):
    """
    Callback para calcular métricas no diferenciables al final de cada época.

    Importante:
    - La loss sigue siendo sparse_categorical_crossentropy.
    - El modelo se selecciona con val_macro_f1.
    - Para diagnosticar overfitting se compara train_macro_f1 vs val_macro_f1.
    """

    def __init__(self, x_train, y_train, x_val, y_val, batch_size=32, every_n_epochs=1):
        super().__init__()
        self.x_train = x_train
        self.y_train = np.asarray(y_train).reshape(-1)
        self.x_val = x_val
        self.y_val = np.asarray(y_val).reshape(-1)
        self.batch_size = batch_size
        self.every_n_epochs = every_n_epochs

    def _predict_metrics(self, x, y):
        y_prob = self.model.predict(x, batch_size=self.batch_size, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        macro_f1 = f1_score(y, y_pred, average="macro", zero_division=0)
        bal_acc = balanced_accuracy_score(y, y_pred)
        kappa = cohen_kappa_score(y, y_pred)

        return macro_f1, bal_acc, kappa

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        if (epoch + 1) % self.every_n_epochs != 0:
            logs.setdefault("train_macro_f1", np.nan)
            logs.setdefault("train_balanced_accuracy", np.nan)
            logs.setdefault("val_macro_f1", np.nan)
            logs.setdefault("val_balanced_accuracy", np.nan)
            logs.setdefault("val_kappa", np.nan)
            return

        train_macro_f1, train_bal_acc, _ = self._predict_metrics(
            self.x_train,
            self.y_train
        )

        val_macro_f1, val_bal_acc, val_kappa = self._predict_metrics(
            self.x_val,
            self.y_val
        )

        logs["train_macro_f1"] = train_macro_f1
        logs["train_balanced_accuracy"] = train_bal_acc
        logs["val_macro_f1"] = val_macro_f1
        logs["val_balanced_accuracy"] = val_bal_acc
        logs["val_kappa"] = val_kappa

        print(
            f" - train_macro_f1: {train_macro_f1:.4f}"
            f" - val_macro_f1: {val_macro_f1:.4f}"
            f" - val_balanced_accuracy: {val_bal_acc:.4f}"
            f" - val_kappa: {val_kappa:.4f}"
        )


def make_keras_callbacks(run_id: str, x_train, y_train, x_val, y_val, batch_size: int, checkpoint_path: str):
    log_path = os.path.join(LOG_DIR, f"{run_id}_training_log.csv")
    monitor = EARLY_STOPPING_MONITOR

    return [
        TrainValMacroF1Callback(
            x_train=x_train,
            y_train=y_train,
            x_val=x_val,
            y_val=y_val,
            batch_size=batch_size,
            every_n_epochs=EVALUATE_VAL_F1_EVERY_N_EPOCHS,
        ),
        CSVLogger(log_path),
        EarlyStopping(
            monitor=monitor,
            mode="max",
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor=monitor,
            mode="max",
            factor=0.3,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor,
            mode="max",
            save_best_only=True,
            verbose=1
        ),
    ]


def metric_row(model_id, display_name, pretraining, tuning, seed, y_true, y_pred, y_prob=None):
    precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        zero_division=0
    )

    row = {
        "model_id": model_id,
        "display_name": display_name,
        "pretraining": pretraining,
        "tuning": tuning,
        "seed": seed,
        "status": "OK",
        "max_epochs": np.nan,
        "early_stopping_monitor": EARLY_STOPPING_MONITOR,
        "checkpoint_monitor": CHECKPOINT_MONITOR,
        "class_weight_strategy": CLASS_WEIGHT_STRATEGY,
        "augmentation_policy": (
            f"horizontal_flip={AUG_HORIZONTAL_FLIP}; "
            f"rotation={AUG_ROTATION_DEGREES}deg; "
            f"zoom={AUG_ZOOM}; "
            f"translation={AUG_TRANSLATION}; "
            f"contrast={AUG_CONTRAST}"
        ),
        "loss_test": np.nan,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision_std_classes": float(np.std(precision_cls)),
        "macro_recall_std_classes": float(np.std(recall_cls)),
        "macro_f1_std_classes": float(np.std(f1_cls)),
    }

    return row


def mark_not_run(model_id: str, display_name: str, reason: str) -> None:
    MODEL_STATUSES[model_id] = f"NOT_RUN: {reason}"
    log_error(f"{model_id} NOT_RUN -> {reason}")
    print(f"[NOT_RUN] {model_id}: {reason}")


def save_history_csv(history_df: pd.DataFrame, run_id: str) -> str:
    path = os.path.join(HISTORIES_DIR, f"{run_id}_history.csv")
    history_df.to_csv(path, index=False)
    return path


def save_predictions_csv(y_true, y_pred, y_prob, run_id: str) -> str:
    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "true_class": [class_names[int(i)] for i in y_true],
        "pred_class": [class_names[int(i)] for i in y_pred],
    })

    if y_prob is not None:
        for i, class_name in enumerate(class_names):
            pred_df[f"prob_{i}_{safe_name(class_name)}"] = y_prob[:, i]

    path = os.path.join(PREDICTIONS_DIR, f"{run_id}_predictions.csv")
    pred_df.to_csv(path, index=False)

    return path


def save_classification_report_csv(y_true, y_pred, run_id: str) -> str:
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
        output_dict=True
    )

    path = os.path.join(CLASS_REPORT_DIR, f"{run_id}_classification_report.csv")
    pd.DataFrame(report).transpose().to_csv(path)

    return path


def save_metrics_json(row: dict, run_id: str) -> str:
    path = os.path.join(METRICS_DIR, f"{run_id}_metrics.json")

    serializable = {
        k: (
            float(v)
            if isinstance(v, (np.floating, np.integer))
            else v
        )
        for k, v in row.items()
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)

    return path


def save_confusion_matrices(y_true, y_pred, run_id: str, title: str) -> tuple[str, str]:
    cm_abs = confusion_matrix(y_true, y_pred)
    cm_norm = cm_abs.astype(float) / np.maximum(cm_abs.sum(axis=1, keepdims=True), 1)

    abs_path = os.path.join(CM_DIR, f"{run_id}_cm_absolute.png")
    norm_path = os.path.join(CM_DIR, f"{run_id}_cm_normalized.png")

    for matrix, path, fmt, suffix in [
        (cm_abs, abs_path, "d", "Absoluta"),
        (cm_norm, norm_path, ".2f", "Normalizada por fila"),
    ]:
        plt.figure(figsize=(9, 7))
        sns.heatmap(
            matrix,
            annot=True,
            fmt=fmt,
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names
        )
        plt.title(f"{title} — {suffix}")
        plt.xlabel("Predicción")
        plt.ylabel("Real")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.savefig(path, dpi=300, bbox_inches="tight")
        plt.close()

    return abs_path, norm_path


def plot_learning_curves(history_df: pd.DataFrame, run_id: str, title: str) -> str:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1) Loss
    if "loss" in history_df:
        axes[0].plot(history_df["loss"], label="train")

    if "val_loss" in history_df:
        axes[0].plot(history_df["val_loss"], label="val")

    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    # 2) Macro F1: train vs validation
    if "train_macro_f1" in history_df:
        axes[1].plot(history_df["train_macro_f1"], label="train")

    if "val_macro_f1" in history_df:
        axes[1].plot(history_df["val_macro_f1"], label="val")

    axes[1].set_title("Macro F1")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    # 3) Métricas principales en validación
    if "val_macro_f1" in history_df:
        axes[2].plot(history_df["val_macro_f1"], label="Val Macro F1")

    if "val_balanced_accuracy" in history_df:
        axes[2].plot(history_df["val_balanced_accuracy"], label="Val Balanced Acc.")

    if "val_kappa" in history_df:
        axes[2].plot(history_df["val_kappa"], label="Val Kappa")

    axes[2].set_title("Validation metrics")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    fig.suptitle(title)
    fig.tight_layout()

    path = os.path.join(LEARNING_CURVES_DIR, f"{run_id}_learning_curves.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return path


def update_best_cache(model_id: str, row: dict, payload: dict) -> None:
    current = BEST_RUN_CACHE.get(model_id)

    if current is None or row["f1_macro"] > current["row"]["f1_macro"]:
        BEST_RUN_CACHE[model_id] = {
            "row": row,
            **payload
        }

## 6. Modelos

Comparamos una CNN propia, dos variantes de ResNet50 con ImageNet, dos variantes con RadImageNet/RAC y un modelo dermatológico EfficientNet-B1 preentrenado en el entorno ISIC.

**Relación con el temario**
- El trabajo aborda clasificación de imágenes a partir de etiquetas por imagen.
- La CNN propia combina Conv2D, BatchNorm, ReLU, MaxPooling, Dropout y softmax.
- ResNet50 ImageNet permite estudiar transferencia generalista.
- RadImageNet/RAC permite estudiar transferencia desde imagen médica no dermatológica.
- EfficientNet-B1 ISIC se usa como referencia dermatológica congelada.
- Grad-CAM se emplea para interpretar visualmente el modelo.


In [ ]:
L2_REG = 1e-4
WEIGHT_DECAY = 1e-4


def build_custom_cnn(input_shape=(224, 224, 3), num_classes=7):
    inputs = keras.Input(shape=input_shape, name="input_image")
    x = layers.Rescaling(1.0 / 255.0)(inputs)
    x = build_augmentation_layer()(x)

    for filters, drop, block_name in [(32, 0.20, "block1"), (64, 0.30, "block2")]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(L2_REG), name=f"{block_name}_conv1")(x)
        x = layers.BatchNormalization(name=f"{block_name}_bn1")(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(L2_REG), name=f"{block_name}_conv2")(x)
        x = layers.BatchNormalization(name=f"{block_name}_bn2")(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop)(x)

    x = layers.Conv2D(128, 3, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(L2_REG), name="last_conv_custom")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="custom_cnn_224")
    return model


def build_resnet50_transfer(input_shape=(224, 224, 3), num_classes=7, model_name="resnet50_transfer"):
    inputs = keras.Input(shape=input_shape, name="input_image")
    x = build_augmentation_layer()(inputs)
    x = layers.Lambda(resnet50_preprocess, name="resnet50_preprocess")(x)
    base_model = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base_model.trainable = False
    x = base_model(x, training=False)
    x = layers.Activation("linear", name="gradcam_resnet50_output")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name=model_name)
    return model, base_model


def freeze_resnet50_backbone(base_model):
    base_model.trainable = False
    for layer in base_model.layers:
        layer.trainable = False


def set_resnet50_conv5_trainable(base_model):
    base_model.trainable = True
    for layer in base_model.layers:
        is_conv5 = layer.name.startswith("conv5_block")
        is_bn = isinstance(layer, layers.BatchNormalization)
        layer.trainable = bool(is_conv5 and not is_bn)


class TorchArrayDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels.astype(int)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.labels[idx])


RADIMAGENET_RESNET50_WEIGHTS_PATH = os.path.join(WEIGHTS_RAD_DIR, "resnet50_torch.pt")
RADIMAGENET_OFFICIAL_REPO = "https://github.com/BMEII-AI/RadImageNet"
RADIMAGENET_PYTORCH_GDRIVE = "https://drive.google.com/file/d/1RHt2GnuOYlc_gcoTETtBDSW73mFyRAtR/view?usp=sharing"
RADIMAGENET_TF_FOLDER = "https://drive.google.com/drive/folders/1Es7cK1hv7zNHJoUW0tI0e6nLFVYTqPqK?usp=sharing"
RADIMAGENET_AUTODOWNLOAD = False

MAXNET_ISIC_WEIGHTS_PATH = os.path.join(WEIGHTS_DERM_DIR, "best_weights.pth")
MAXNET_ISIC_HF_REPO = "conan17970/efficientnet-b1-skin-cancer-isic2019"
MAXNET_ISIC_HF_FILE = "best_weights.pth"


def ensure_radimagenet_weights():
    if os.path.exists(RADIMAGENET_RESNET50_WEIGHTS_PATH):
        return RADIMAGENET_RESNET50_WEIGHTS_PATH
    if RADIMAGENET_AUTODOWNLOAD:
        print("Se solicita descarga automática, pero por robustez la ruta oficial se deja como subida manual a Drive.")
    print("No se han encontrado pesos RadImageNet/RAC.")
    print("Sube manualmente `resnet50_torch.pt` a:", RADIMAGENET_RESNET50_WEIGHTS_PATH)
    print("Fuente oficial:", RADIMAGENET_OFFICIAL_REPO)
    return None


class RadImageNetResNet50(nn.Module):
    def __init__(self, num_classes=7, dropout=0.4, hidden_units=256):
        super().__init__()
        self.backbone = torch_resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, hidden_units),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        if x.ndim > 2:
            x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


def _extract_checkpoint_state_dict(payload):
    if isinstance(payload, dict):
        for key in ("state_dict", "model_state_dict", "model", "backbone"):
            nested = payload.get(key)
            if isinstance(nested, dict):
                return nested
        if all(torch.is_tensor(v) for v in payload.values()):
            return payload
    if isinstance(payload, dict):
        return payload
    raise TypeError("El checkpoint cargado no contiene un state_dict compatible.")


def _strip_checkpoint_prefixes(state_dict):
    cleaned = {}
    prefixes = ("module.", "model.", "backbone.")
    for key, value in state_dict.items():
        new_key = key
        prefix_removed = True
        while prefix_removed:
            prefix_removed = False
            for prefix in prefixes:
                if new_key.startswith(prefix):
                    new_key = new_key[len(prefix):]
                    prefix_removed = True
        cleaned[new_key] = value
    return cleaned


def load_radimagenet_resnet50_classifier(weights_path: str, num_classes=7):
    model = RadImageNetResNet50(num_classes=num_classes)
    payload = torch.load(weights_path, map_location="cpu")
    state_dict = _extract_checkpoint_state_dict(payload)
    state_dict = _strip_checkpoint_prefixes(state_dict)
    backbone_state = {k: v for k, v in state_dict.items() if not k.startswith("classifier.")}
    incompatible = model.backbone.load_state_dict(backbone_state, strict=False)
    print(
        "Carga RadImageNet/RAC -> missing:",
        len(incompatible.missing_keys),
        "| unexpected:",
        len(incompatible.unexpected_keys),
    )
    return model


def freeze_rad_backbone(model: nn.Module):
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def set_rad_layer4_trainable(model: nn.Module):
    freeze_rad_backbone(model)
    for param in model.backbone.layer4.parameters():
        param.requires_grad = True
    for module in model.backbone.layer4.modules():
        if isinstance(module, nn.BatchNorm2d):
            for param in module.parameters():
                param.requires_grad = False
            module.eval()


def freeze_batchnorm_eval(model: nn.Module):
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)):
            module.eval()
            for param in module.parameters():
                param.requires_grad = False


def ensure_isic_weights():
    if os.path.exists(MAXNET_ISIC_WEIGHTS_PATH):
        return MAXNET_ISIC_WEIGHTS_PATH
    try:
        downloaded = hf_hub_download(
            repo_id=MAXNET_ISIC_HF_REPO,
            filename=MAXNET_ISIC_HF_FILE,
            local_dir=WEIGHTS_DERM_DIR,
        )
        return downloaded
    except Exception as exc:
        log_error(f"No se pudieron descargar pesos ISIC/EfficientNet-B1: {exc}")
        print("No se pudieron descargar los pesos ISIC/EfficientNet-B1.")
        print("M6 se marcará como NOT_RUN si no subes manualmente:", MAXNET_ISIC_WEIGHTS_PATH)
        return None


def build_effnet_isic_baseline(weights_path: str, num_classes=7):
    model = timm.create_model("efficientnet_b1", pretrained=False, num_classes=6)
    state_dict = torch.load(weights_path, map_location="cpu")
    if isinstance(state_dict, dict) and "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model


print("Modelos y rutas opcionales configurados.")


## 7. Entrenamiento

Entrenamos cada modelo con la misma estrategia general de evaluación, pesos de clase y callbacks. La comparación entre shallow y deep parcial mantiene el mismo máximo de épocas.


In [ ]:
print("class_weights_main:", class_weights_main)
print("Estrategia:", CLASS_WEIGHT_STRATEGY)

# Comprobación visual de que son suavizados
assert max(class_weights_main.values()) <= 2.5
assert min(class_weights_main.values()) >= 0.7
print("Pesos suavizados OK.")

In [ ]:
def run_keras_experiment(
    model,
    model_id,
    display_name,
    pretraining,
    tuning,
    seed,
    x_train,
    y_train,
    x_val,
    y_val,
    x_test,
    y_test,
    class_weights,
    learning_rate,
    epochs,
    optimizer_name="adamw",
):
    run_id = f"{model_id}_seed{seed}"
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{run_id}_best.keras")
    final_path = os.path.join(FINAL_MODEL_DIR, f"{run_id}_final.keras")

    callbacks = make_keras_callbacks(
        run_id,
        x_train,
        y_train,
        x_val,
        y_val,
        batch_size=BATCH_SIZE,
        checkpoint_path=checkpoint_path,
    )

    if optimizer_name == "adamw":
        optimizer = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=WEIGHT_DECAY
        )
    else:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=[]
    )

    start = time.time()

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )

    train_time = time.time() - start
    model.save(final_path)

    history_df = pd.DataFrame(history.history)
    history_path = save_history_csv(history_df, run_id)
    curve_path = plot_learning_curves(history_df, run_id, display_name)

    y_prob = model.predict(x_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    row = metric_row(
        model_id,
        display_name,
        pretraining,
        tuning,
        seed,
        y_test,
        y_pred,
        y_prob
    )

    row["max_epochs"] = int(epochs)
    row["loss_test"] = float(
        keras.losses.sparse_categorical_crossentropy(y_test, y_prob).numpy().mean()
    )
    row["n_params_total"] = int(model.count_params())
    row["n_params_trainable"] = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
    row["training_time_sec"] = float(train_time)
    row["best_epoch"] = int(np.nanargmax(history_df["val_macro_f1"].values) + 1)
    row["best_val_macro_f1"] = float(np.nanmax(history_df["val_macro_f1"].values))
    row["best_val_loss"] = float(np.nanmin(history_df["val_loss"].values))
    row["checkpoint_path"] = checkpoint_path
    row["final_model_path"] = final_path
    row["history_path"] = history_path
    row["curve_path"] = curve_path

    save_predictions_csv(y_test, y_pred, y_prob, run_id)
    save_classification_report_csv(y_test, y_pred, run_id)
    save_metrics_json(row, run_id)
    save_confusion_matrices(y_test, y_pred, run_id, display_name)

    ALL_RESULTS.append(row)
    MODEL_STATUSES[model_id] = "OK"

    update_best_cache(
        model_id,
        row,
        {
            "family": "keras",
            "model": model,
            "x_test": x_test,
            "y_true": y_test,
            "y_pred": y_pred,
            "y_prob": y_prob,
        }
    )

    return row

if RUN_M1:
    for seed in SEEDS:
        print(f"\n===== Ejecutando M1 seed={seed} =====")
        set_seed(seed)
        model = build_custom_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES)
        run_keras_experiment(
            model=model,
            model_id="M1_CNN_PROPIA_224",
            display_name="M1: CNN propia",
            pretraining="No",
            tuning="Entrenamiento completo",
            seed=seed,
            x_train=X_train_raw,
            y_train=y_train,
            x_val=X_val_raw,
            y_val=y_val,
            x_test=X_test_raw,
            y_test=y_test,
            class_weights=class_weights_main,
            learning_rate=LR_HEAD,
            epochs=EPOCHS_HEAD,
        )
else:
    mark_not_run("M1_CNN_PROPIA_224", "M1: CNN propia", "Desactivado por configuración")

### ResNet50 ImageNet


In [ ]:
# ============================================================
# ResNet50 ImageNet: shallow y deep como experimentos independientes
# ============================================================

if RUN_M2:
    for seed in SEEDS:
        print(f"\n===== Ejecutando M2 ImageNet shallow seed={seed} =====")
        set_seed(seed)

        shallow_model, shallow_base = build_resnet50_transfer(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            num_classes=NUM_CLASSES,
            model_name=f"m2_resnet50_imagenet_shallow_seed{seed}",
        )

        freeze_resnet50_backbone(shallow_base)

        run_keras_experiment(
            model=shallow_model,
            model_id="M2_RESNET50_IMAGENET_SHALLOW",
            display_name="M2: ResNet50 ImageNet shallow",
            pretraining="ImageNet",
            tuning="Shallow tuning",
            seed=seed,
            x_train=X_train_raw,
            y_train=y_train,
            x_val=X_val_raw,
            y_val=y_val,
            x_test=X_test_raw,
            y_test=y_test,
            class_weights=class_weights_main,
            learning_rate=LR_HEAD,
            epochs=EPOCHS_HEAD,
        )


if RUN_M3:
    for seed in SEEDS:
        print(f"\n===== Ejecutando M3 ImageNet deep directo seed={seed} =====")
        set_seed(seed)

        deep_model, deep_base = build_resnet50_transfer(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            num_classes=NUM_CLASSES,
            model_name=f"m3_resnet50_imagenet_deep_seed{seed}",
        )

        # IMPORTANTE:
        # Protocolo tipo práctica/docencia.
        # Se parte directamente de ResNet50(weights='imagenet'),
        # se entrena la cabeza nueva y se desbloquea el último bloque residual.
        # NO se carga el checkpoint shallow M2.
        set_resnet50_conv5_trainable(deep_base)

        run_keras_experiment(
            model=deep_model,
            model_id="M3_RESNET50_IMAGENET_DEEP",
            display_name="M3: ResNet50 ImageNet deep",
            pretraining="ImageNet",
            tuning="Deep tuning parcial: cabeza + último bloque residual",
            seed=seed,
            x_train=X_train_raw,
            y_train=y_train,
            x_val=X_val_raw,
            y_val=y_val,
            x_test=X_test_raw,
            y_test=y_test,
            class_weights=class_weights_main,
            learning_rate=LR_DEEP,
            epochs=EPOCHS_DEEP,
        )

### ResNet50 RadImageNet/RAC


In [ ]:
# ============================================================
# Loader correcto para RadImageNet/RAC ResNet50 PyTorch
# Convierte checkpoint tipo Sequential:
# backbone.0 -> conv1
# backbone.1 -> bn1
# backbone.4 -> layer1
# backbone.5 -> layer2
# backbone.6 -> layer3
# backbone.7 -> layer4
# ============================================================

def _extract_checkpoint_state_dict(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()

    if isinstance(payload, dict):
        for key in [
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "network",
            "weights",
            "params",
        ]:
            value = payload.get(key)
            if isinstance(value, dict):
                return value

        tensor_items = {k: v for k, v in payload.items() if torch.is_tensor(v)}
        if tensor_items:
            return tensor_items

    raise TypeError("No se pudo extraer un state_dict compatible del checkpoint.")


def _map_radimagenet_key_to_torchvision(key: str):
    """
    Mapea claves del checkpoint oficial RadImageNet/RAC a torchvision ResNet50.

    Ejemplos:
    backbone.0.weight -> conv1.weight
    backbone.1.weight -> bn1.weight
    backbone.4.0.conv1.weight -> layer1.0.conv1.weight
    backbone.5.0.conv1.weight -> layer2.0.conv1.weight
    backbone.6.0.conv1.weight -> layer3.0.conv1.weight
    backbone.7.0.conv1.weight -> layer4.0.conv1.weight
    """

    # Quitar prefijos típicos
    prefixes = ["module.", "model.", "net.", "network."]
    for prefix in prefixes:
        if key.startswith(prefix):
            key = key[len(prefix):]

    # Caso específico del checkpoint RadImageNet oficial
    mapping = {
        "backbone.0.": "conv1.",
        "backbone.1.": "bn1.",
        "backbone.4.": "layer1.",
        "backbone.5.": "layer2.",
        "backbone.6.": "layer3.",
        "backbone.7.": "layer4.",
    }

    for old_prefix, new_prefix in mapping.items():
        if key.startswith(old_prefix):
            return new_prefix + key[len(old_prefix):]

    # Ignorar capas sin parámetros o cabeza original si apareciesen
    ignored_prefixes = [
        "backbone.2.",   # relu
        "backbone.3.",   # maxpool
        "backbone.8.",   # avgpool
        "backbone.9.",   # fc
        "fc.",
        "classifier.",
    ]

    for ignored in ignored_prefixes:
        if key.startswith(ignored):
            return None

    return key


def convert_radimagenet_state_dict_to_torchvision(raw_state_dict, target_state_dict):
    converted = {}
    skipped = []
    shape_mismatch = []

    for raw_key, value in raw_state_dict.items():
        if not torch.is_tensor(value):
            continue

        new_key = _map_radimagenet_key_to_torchvision(raw_key)

        if new_key is None:
            skipped.append(raw_key)
            continue

        if new_key not in target_state_dict:
            skipped.append(raw_key)
            continue

        if tuple(value.shape) != tuple(target_state_dict[new_key].shape):
            shape_mismatch.append(
                (raw_key, new_key, tuple(value.shape), tuple(target_state_dict[new_key].shape))
            )
            continue

        converted[new_key] = value

    return converted, skipped, shape_mismatch


def load_radimagenet_resnet50_classifier(weights_path: str, num_classes=7):
    """
    Carga una ResNet50 torchvision con pesos RadImageNet/RAC en el backbone
    y mantiene una cabeza nueva para DermaMNIST.
    """
    model = RadImageNetResNet50(num_classes=num_classes)

    print("Cargando checkpoint RadImageNet/RAC desde:")
    print(weights_path)

    payload = torch.load(weights_path, map_location="cpu")
    raw_state_dict = _extract_checkpoint_state_dict(payload)

    print("Número de tensores en checkpoint:", len(raw_state_dict))
    print("Primeras claves checkpoint:")
    for k in list(raw_state_dict.keys())[:12]:
        print("  ", k)

    target_state = model.backbone.state_dict()

    converted_state, skipped, shape_mismatch = convert_radimagenet_state_dict_to_torchvision(
        raw_state_dict,
        target_state
    )

    expected_keys = list(target_state.keys())

    print("\nTensores esperados backbone:", len(expected_keys))
    print("Tensores convertidos/emparejados:", len(converted_state))

    coverage = len(converted_state) / max(1, len(expected_keys))
    print(f"Cobertura de carga backbone: {coverage:.1%}")

    if shape_mismatch:
        print("\nShape mismatches detectados:")
        for item in shape_mismatch[:10]:
            print(item)

    if skipped:
        print("\nClaves saltadas:", len(skipped))
        print("Primeras claves saltadas:")
        for k in skipped[:10]:
            print("  ", k)

    incompatible = model.backbone.load_state_dict(converted_state, strict=False)

    print(
        "\nCarga RadImageNet/RAC corregida -> missing:",
        len(incompatible.missing_keys),
        "| unexpected:",
        len(incompatible.unexpected_keys),
    )

    print("Primeras missing:")
    print(incompatible.missing_keys[:10])

    print("Primeras unexpected:")
    print(incompatible.unexpected_keys[:10])

    if coverage < 0.90:
        print("\nADVERTENCIA:")
        print("La cobertura sigue siendo baja. No conviene usar M4/M5 sin revisar.")
    else:
        print("\nOK: pesos RadImageNet/RAC cargados correctamente en el backbone.")

    return model

In [ ]:
rad_weights_path = ensure_radimagenet_weights()

tmp_model = load_radimagenet_resnet50_classifier(
    rad_weights_path,
    num_classes=NUM_CLASSES
)

In [ ]:
rad_train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5 if AUG_HORIZONTAL_FLIP else 0.0),
    transforms.RandomRotation(AUG_ROTATION_DEGREES),
    transforms.ColorJitter(brightness=AUG_TORCH_COLOR_JITTER, contrast=AUG_TORCH_COLOR_JITTER, saturation=AUG_TORCH_COLOR_JITTER),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

rad_eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


def train_torch_classifier(
    model,
    model_id,
    display_name,
    pretraining,
    tuning,
    seed,
    train_loader,
    val_loader,
    test_loader,
    class_weights,
    learning_rate,
    epochs,
    checkpoint_ext="pt",
    freeze_batchnorm_stats=False,
    denorm_mode="minus_one_one",
):
    run_id = f"{model_id}_seed{seed}"
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{run_id}_best.{checkpoint_ext}")
    final_path = os.path.join(FINAL_MODEL_DIR, f"{run_id}_final.{checkpoint_ext}")

    weights = torch.tensor([class_weights[i] for i in range(NUM_CLASSES)], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=3, min_lr=1e-6)

    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_f1": [],
        "val_balanced_accuracy": [],
        "val_kappa": [],
        "learning_rate": [],
    }
    best_macro_f1 = -np.inf
    best_epoch = 0
    epochs_without_improvement = 0
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_batchnorm_stats:
            freeze_batchnorm_eval(model)
        train_losses = []
        train_preds, train_labels = [], []

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_preds.append(torch.argmax(logits, dim=1).detach().cpu().numpy())
            train_labels.append(labels.detach().cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)
        train_loss = float(np.mean(train_losses))
        train_acc = accuracy_score(train_labels, train_preds)

        model.eval()
        val_losses, val_probs, val_preds, val_labels = [], [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                logits = model(images)
                loss = criterion(logits, labels)
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)
                val_losses.append(loss.item())
                val_probs.append(probs.cpu().numpy())
                val_preds.append(preds.cpu().numpy())
                val_labels.append(labels.cpu().numpy())

        y_val_prob = np.concatenate(val_probs)
        y_val_pred = np.concatenate(val_preds)
        y_val_true = np.concatenate(val_labels)
        val_macro_f1 = f1_score(y_val_true, y_val_pred, average="macro", zero_division=0)
        val_bal_acc = balanced_accuracy_score(y_val_true, y_val_pred)
        val_kappa = cohen_kappa_score(y_val_true, y_val_pred)
        val_loss = float(np.mean(val_losses))

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(accuracy_score(y_val_true, y_val_pred))
        history["val_macro_f1"].append(val_macro_f1)
        history["val_balanced_accuracy"].append(val_bal_acc)
        history["val_kappa"].append(val_kappa)
        history["learning_rate"].append(optimizer.param_groups[0]["lr"])

        print(
            f"Epoch {epoch:02d} | loss={train_loss:.4f} | acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_macro_f1={val_macro_f1:.4f} | "
            f"val_bal_acc={val_bal_acc:.4f} | val_kappa={val_kappa:.4f}"
        )

        scheduler.step(val_macro_f1)
        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= 10:
            print("Early stopping activado.")
            break

    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    torch.save(model.state_dict(), final_path)

    model.eval()
    test_losses, probs_all, preds_all, labels_all = [], [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            test_losses.append(loss.item())
            probs_all.append(probs.cpu().numpy())
            preds_all.append(preds.cpu().numpy())
            labels_all.append(labels.cpu().numpy())

    y_prob = np.concatenate(probs_all)
    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(labels_all)
    history_df = pd.DataFrame(history)
    save_history_csv(history_df, run_id)
    plot_learning_curves(history_df, run_id, display_name)

    row = metric_row(model_id, display_name, pretraining, tuning, seed, y_true, y_pred, y_prob)
    row["max_epochs"] = int(epochs)
    row["loss_test"] = float(np.mean(test_losses))
    row["n_params_total"] = int(sum(p.numel() for p in model.parameters()))
    row["n_params_trainable"] = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    row["training_time_sec"] = float(time.time() - start)
    row["best_epoch"] = int(best_epoch)
    row["best_val_macro_f1"] = float(best_macro_f1)
    row["best_val_loss"] = float(np.nanmin(history_df["val_loss"].values))
    row["checkpoint_path"] = checkpoint_path
    row["final_model_path"] = final_path

    save_predictions_csv(y_true, y_pred, y_prob, run_id)
    save_classification_report_csv(y_true, y_pred, run_id)
    save_metrics_json(row, run_id)
    save_confusion_matrices(y_true, y_pred, run_id, display_name)
    ALL_RESULTS.append(row)
    MODEL_STATUSES[model_id] = "OK"
    update_best_cache(model_id, row, {
        "family": "torch",
        "model": model,
        "test_loader": test_loader,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "denorm": denorm_mode,
    })
    return row





In [ ]:
rad_train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5 if AUG_HORIZONTAL_FLIP else 0.0),
    transforms.RandomRotation(AUG_ROTATION_DEGREES),
    transforms.ColorJitter(brightness=AUG_TORCH_COLOR_JITTER, contrast=AUG_TORCH_COLOR_JITTER, saturation=AUG_TORCH_COLOR_JITTER),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

rad_eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


def train_torch_classifier(
    model,
    model_id,
    display_name,
    pretraining,
    tuning,
    seed,
    train_loader,
    val_loader,
    test_loader,
    class_weights,
    learning_rate,
    epochs,
    checkpoint_ext="pt",
    freeze_batchnorm_stats=False,
    denorm_mode="minus_one_one",
):
    run_id = f"{model_id}_seed{seed}"
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{run_id}_best.{checkpoint_ext}")
    final_path = os.path.join(FINAL_MODEL_DIR, f"{run_id}_final.{checkpoint_ext}")

    weights = torch.tensor([class_weights[i] for i in range(NUM_CLASSES)], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=3, min_lr=1e-6)

    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_f1": [],
        "val_balanced_accuracy": [],
        "val_kappa": [],
        "learning_rate": [],
    }
    best_macro_f1 = -np.inf
    best_epoch = 0
    epochs_without_improvement = 0
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_batchnorm_stats:
            freeze_batchnorm_eval(model)
        train_losses = []
        train_preds, train_labels = [], []

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            train_preds.append(torch.argmax(logits, dim=1).detach().cpu().numpy())
            train_labels.append(labels.detach().cpu().numpy())

        train_preds = np.concatenate(train_preds)
        train_labels = np.concatenate(train_labels)
        train_loss = float(np.mean(train_losses))
        train_acc = accuracy_score(train_labels, train_preds)

        model.eval()
        val_losses, val_probs, val_preds, val_labels = [], [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                logits = model(images)
                loss = criterion(logits, labels)
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)
                val_losses.append(loss.item())
                val_probs.append(probs.cpu().numpy())
                val_preds.append(preds.cpu().numpy())
                val_labels.append(labels.cpu().numpy())

        y_val_prob = np.concatenate(val_probs)
        y_val_pred = np.concatenate(val_preds)
        y_val_true = np.concatenate(val_labels)
        val_macro_f1 = f1_score(y_val_true, y_val_pred, average="macro", zero_division=0)
        val_bal_acc = balanced_accuracy_score(y_val_true, y_val_pred)
        val_kappa = cohen_kappa_score(y_val_true, y_val_pred)
        val_loss = float(np.mean(val_losses))

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(accuracy_score(y_val_true, y_val_pred))
        history["val_macro_f1"].append(val_macro_f1)
        history["val_balanced_accuracy"].append(val_bal_acc)
        history["val_kappa"].append(val_kappa)
        history["learning_rate"].append(optimizer.param_groups[0]["lr"])

        print(
            f"Epoch {epoch:02d} | loss={train_loss:.4f} | acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_macro_f1={val_macro_f1:.4f} | "
            f"val_bal_acc={val_bal_acc:.4f} | val_kappa={val_kappa:.4f}"
        )

        scheduler.step(val_macro_f1)
        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= 10:
            print("Early stopping activado.")
            break

    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    torch.save(model.state_dict(), final_path)

    model.eval()
    test_losses, probs_all, preds_all, labels_all = [], [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            test_losses.append(loss.item())
            probs_all.append(probs.cpu().numpy())
            preds_all.append(preds.cpu().numpy())
            labels_all.append(labels.cpu().numpy())

    y_prob = np.concatenate(probs_all)
    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(labels_all)
    history_df = pd.DataFrame(history)
    save_history_csv(history_df, run_id)
    plot_learning_curves(history_df, run_id, display_name)

    row = metric_row(model_id, display_name, pretraining, tuning, seed, y_true, y_pred, y_prob)
    row["max_epochs"] = int(epochs)
    row["loss_test"] = float(np.mean(test_losses))
    row["n_params_total"] = int(sum(p.numel() for p in model.parameters()))
    row["n_params_trainable"] = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    row["training_time_sec"] = float(time.time() - start)
    row["best_epoch"] = int(best_epoch)
    row["best_val_macro_f1"] = float(best_macro_f1)
    row["best_val_loss"] = float(np.nanmin(history_df["val_loss"].values))
    row["checkpoint_path"] = checkpoint_path
    row["final_model_path"] = final_path

    save_predictions_csv(y_true, y_pred, y_prob, run_id)
    save_classification_report_csv(y_true, y_pred, run_id)
    save_metrics_json(row, run_id)
    save_confusion_matrices(y_true, y_pred, run_id, display_name)
    ALL_RESULTS.append(row)
    MODEL_STATUSES[model_id] = "OK"
    update_best_cache(model_id, row, {
        "family": "torch",
        "model": model,
        "test_loader": test_loader,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "denorm": denorm_mode,
    })
    return row


# ============================================================
# ResNet50 RadImageNet/RAC: shallow y deep independientes
# ============================================================

rad_weights_path = ensure_radimagenet_weights()

if rad_weights_path is not None:
    for seed in SEEDS:
        set_seed(seed)

        train_ds_rad = TorchArrayDataset(X_train_raw, y_train, transform=rad_train_tfms)
        val_ds_rad = TorchArrayDataset(X_val_raw, y_val, transform=rad_eval_tfms)
        test_ds_rad = TorchArrayDataset(X_test_raw, y_test, transform=rad_eval_tfms)

        train_loader_rad = DataLoader(
            train_ds_rad,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
        )

        val_loader_rad = DataLoader(
            val_ds_rad,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
        )

        test_loader_rad = DataLoader(
            test_ds_rad,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
        )

        if RUN_M4:
            print(f"\n===== Ejecutando M4 RAC shallow seed={seed} =====")

            model_m4 = load_radimagenet_resnet50_classifier(
                rad_weights_path,
                num_classes=NUM_CLASSES,
            ).to(device)

            freeze_rad_backbone(model_m4)

            train_torch_classifier(
                model=model_m4,
                model_id="M4_RESNET50_RADIMAGENET_RAC_SHALLOW",
                display_name="M4: ResNet50 RadImageNet/RAC shallow",
                pretraining="RadImageNet/RAC",
                tuning="Shallow tuning",
                seed=seed,
                train_loader=train_loader_rad,
                val_loader=val_loader_rad,
                test_loader=test_loader_rad,
                class_weights=class_weights_main,
                learning_rate=LR_HEAD,
                epochs=EPOCHS_HEAD,
                freeze_batchnorm_stats=True,
                denorm_mode="minus_one_one",
            )

        if RUN_M5:
            print(f"\n===== Ejecutando M5 RAC deep directo seed={seed} =====")

            model_m5 = load_radimagenet_resnet50_classifier(
                rad_weights_path,
                num_classes=NUM_CLASSES,
            ).to(device)

            # IMPORTANTE:
            # Protocolo tipo práctica/docencia.
            # Se parte directamente de los pesos RadImageNet/RAC,
            # se entrena la cabeza nueva y se desbloquea layer4.
            # NO se carga el checkpoint shallow M4.
            set_rad_layer4_trainable(model_m5)

            train_torch_classifier(
                model=model_m5,
                model_id="M5_RESNET50_RADIMAGENET_RAC_DEEP",
                display_name="M5: ResNet50 RadImageNet/RAC deep",
                pretraining="RadImageNet/RAC",
                tuning="Deep tuning parcial: cabeza + último bloque residual",
                seed=seed,
                train_loader=train_loader_rad,
                val_loader=val_loader_rad,
                test_loader=test_loader_rad,
                class_weights=class_weights_main,
                learning_rate=LR_DEEP,
                epochs=EPOCHS_DEEP,
                freeze_batchnorm_stats=True,
                denorm_mode="minus_one_one",
            )

else:
    mark_not_run(
        "M4_RESNET50_RADIMAGENET_RAC_SHALLOW",
        "M4: ResNet50 RadImageNet/RAC shallow",
        "Pesos RadImageNet/RAC no disponibles",
    )

    mark_not_run(
        "M5_RESNET50_RADIMAGENET_RAC_DEEP",
        "M5: ResNet50 RadImageNet/RAC deep",
        "Pesos RadImageNet/RAC no disponibles",
    )


### EfficientNet-B1 ISIC congelado


In [ ]:
effnet_train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5 if AUG_HORIZONTAL_FLIP else 0.0),
    transforms.RandomRotation(AUG_ROTATION_DEGREES),
    transforms.ColorJitter(brightness=AUG_TORCH_COLOR_JITTER, contrast=AUG_TORCH_COLOR_JITTER, saturation=AUG_TORCH_COLOR_JITTER),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

effnet_eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if RUN_M6:
    effnet_weights_path = ensure_isic_weights()
    if effnet_weights_path is None:
        mark_not_run(
            "M6_MAXNET_ISIC_DERMATOLOGY_FROZEN_BASELINE",
            "M6: EfficientNet-B1 ISIC frozen",
            "Pesos ISIC/EfficientNet-B1 no disponibles",
        )
    else:
        for seed in SEEDS:
            set_seed(seed)
            train_ds_eff = TorchArrayDataset(X_train_raw, y_train, transform=effnet_train_tfms)
            val_ds_eff = TorchArrayDataset(X_val_raw, y_val, transform=effnet_eval_tfms)
            test_ds_eff = TorchArrayDataset(X_test_raw, y_test, transform=effnet_eval_tfms)
            train_loader_eff = DataLoader(train_ds_eff, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
            val_loader_eff = DataLoader(val_ds_eff, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            test_loader_eff = DataLoader(test_ds_eff, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

            model_m6 = build_effnet_isic_baseline(effnet_weights_path, num_classes=NUM_CLASSES).to(device)
            train_torch_classifier(
                model=model_m6,
                model_id="M6_MAXNET_ISIC_DERMATOLOGY_FROZEN_BASELINE",
                display_name="M6: EfficientNet-B1 ISIC frozen",
                pretraining="ISIC2019 / HAM10000-proximal",
                tuning="Backbone congelado + top lineal",
                seed=seed,
                train_loader=train_loader_eff,
                val_loader=val_loader_eff,
                test_loader=test_loader_eff,
                class_weights=class_weights_main,
                learning_rate=LR_HEAD,
                epochs=EPOCHS_HEAD,
                freeze_batchnorm_stats=True,
                denorm_mode="imagenet",
            )
else:
    mark_not_run("M6_MAXNET_ISIC_DERMATOLOGY_FROZEN_BASELINE", "M6: EfficientNet-B1 ISIC frozen", "Desactivado por configuración")


## 8. Evaluación

La evaluación en test se realiza una sola vez por seed, cargando siempre el mejor checkpoint según `val_macro_f1`.


In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)
if results_df.empty:
    raise RuntimeError("No se ha ejecutado ningún modelo.")

display(results_df.sort_values(["model_id", "seed"]).reset_index(drop=True))
results_df.to_csv(os.path.join(SUMMARY_DIR, "all_results_long.csv"), index=False)
print("Resultados largos guardados en statistical_summary/all_results_long.csv")

## 9. Comparación de modelos

Resumimos los resultados como media ± desviación típica a partir de las semillas definidas en la configuración.


In [ ]:
numeric_cols = [
    "loss_test", "accuracy", "balanced_accuracy", "precision_macro", "recall_macro",
    "f1_macro", "cohen_kappa", "precision_weighted", "recall_weighted", "f1_weighted",
    "training_time_sec", "best_epoch", "best_val_macro_f1", "best_val_loss",
]

summary_rows = []
for model_id, group in results_df.groupby("model_id"):
    first = group.iloc[0]
    row = {
        "model_id": model_id,
        "display_name": MODEL_DISPLAY_NAMES.get(model_id, first["display_name"]),
        "pretraining": first["pretraining"],
        "tuning": first["tuning"],
        "n_runs": len(group),
        "status": first["status"],
        "max_epochs": first.get("max_epochs", np.nan),
        "early_stopping_monitor": first.get("early_stopping_monitor", EARLY_STOPPING_MONITOR),
        "checkpoint_monitor": first.get("checkpoint_monitor", CHECKPOINT_MONITOR),
        "class_weight_strategy": first.get("class_weight_strategy", CLASS_WEIGHT_STRATEGY),
        "augmentation_policy": first.get("augmentation_policy", ""),
        "model_order": MODEL_ORDER.index(model_id) if model_id in MODEL_ORDER else 999,
    }
    for col in numeric_cols:
        row[f"{col}_mean"] = group[col].mean()
        row[f"{col}_std"] = group[col].std(ddof=0)
        row[f"{col}_mean_pm_std"] = f"{group[col].mean():.3f} ± {group[col].std(ddof=0):.3f}"
    summary_rows.append(row)

summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(["model_order", "display_name"])
    .reset_index(drop=True)
)

display(summary_df[[
    "display_name", "f1_macro_mean_pm_std", "balanced_accuracy_mean_pm_std",
    "cohen_kappa_mean_pm_std", "accuracy_mean_pm_std", "status"
]])

all_results_long_csv = os.path.join(SUMMARY_DIR, "all_results_long.csv")
summary_csv = os.path.join(SUMMARY_DIR, "summary_mean_std.csv")
summary_xlsx = os.path.join(SUMMARY_DIR, "summary_mean_std.xlsx")
results_df.to_csv(all_results_long_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
with pd.ExcelWriter(summary_xlsx) as writer:
    results_df.to_excel(writer, sheet_name="all_results_long", index=False)
    summary_df.to_excel(writer, sheet_name="summary_mean_std", index=False)
print("Resumen guardado en:", summary_csv, "y", summary_xlsx)


### Figuras comparativas


In [ ]:
def ordered_summary_frame():
    df = summary_df.copy()
    df["model_order"] = df["model_id"].map(lambda x: MODEL_ORDER.index(x) if x in MODEL_ORDER else 999)
    return df.sort_values(["model_order", "display_name"]).reset_index(drop=True)


def plot_summary_metric(metric_base: str, title: str, filename: str, color: str):
    df = ordered_summary_frame()
    mean_col = f"{metric_base}_mean"
    std_col = f"{metric_base}_std"
    plt.figure(figsize=(12, 5))
    bars = plt.bar(df["display_name"], df[mean_col], yerr=df[std_col], color=color, capsize=5)
    plt.xticks(rotation=20, ha="right")
    plt.title(title)
    plt.ylabel(metric_base)
    plt.ylim(0, min(1.0, max(df[mean_col].max() + 0.1, 0.2)))
    for bar, (_, row) in zip(bars, df.iterrows()):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            row[f"{metric_base}_mean_pm_std"],
            ha="center",
            va="bottom",
            fontsize=8,
        )
    plt.tight_layout()
    png_path = os.path.join(MODEL_COMPARISON_DIR, f"{filename}.png")
    pdf_path = os.path.join(MODEL_COMPARISON_DIR, f"{filename}.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_combined_main_metrics():
    metrics = [
        ("f1_macro", "Macro F1", PALETTE["primary"]),
        ("balanced_accuracy", "Balanced accuracy", PALETTE["secondary"]),
        ("cohen_kappa", "Cohen kappa", PALETTE["accent"]),
    ]
    df = ordered_summary_frame()
    fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=False)
    for ax, (metric_base, title, color) in zip(axes, metrics):
        mean_col = f"{metric_base}_mean"
        std_col = f"{metric_base}_std"
        bars = ax.bar(df["display_name"], df[mean_col], yerr=df[std_col], color=color, capsize=5)
        ax.set_title(title)
        ax.set_xticks(range(len(df)))
        ax.set_xticklabels(df["display_name"], rotation=25, ha="right")
        ymax = min(1.0, max(df[mean_col].max() + 0.1, 0.2))
        ax.set_ylim(0, ymax)
        for bar, (_, row) in zip(bars, df.iterrows()):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                row[f"{metric_base}_mean_pm_std"],
                ha="center",
                va="bottom",
                fontsize=7,
            )
    fig.suptitle("Comparación principal de modelos", fontsize=14)
    fig.tight_layout()
    png_path = os.path.join(MODEL_COMPARISON_DIR, "combined_main_metrics.png")
    pdf_path = os.path.join(MODEL_COMPARISON_DIR, "combined_main_metrics.pdf")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.show()


plot_summary_metric("f1_macro", "Macro F1: media ± desviación típica", "macro_f1_mean_std", PALETTE["primary"])
plot_summary_metric("balanced_accuracy", "Balanced accuracy: media ± desviación típica", "balanced_accuracy_mean_std", PALETTE["secondary"])
plot_summary_metric("cohen_kappa", "Cohen kappa: media ± desviación típica", "kappa_mean_std", PALETTE["accent"])
plot_combined_main_metrics()


### Análisis de errores


In [ ]:
def save_correct_incorrect_examples(raw_images, y_true, y_pred, y_prob, run_id, n_each=3):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    correct_idx = np.where(y_true == y_pred)[0][:n_each]
    wrong_idx = np.where(y_true != y_pred)[0][:n_each]
    selected = list(correct_idx) + list(wrong_idx)
    tags = ["Correcta"] * len(correct_idx) + ["Error"] * len(wrong_idx)
    if not selected:
        return None
    plt.figure(figsize=(12, 4 * len(selected)))
    for row_id, idx in enumerate(selected):
        plt.subplot(len(selected), 1, row_id + 1)
        plt.imshow(raw_images[idx])
        prob = float(np.max(y_prob[idx])) if y_prob is not None else np.nan
        plt.title(
            f"{tags[row_id]} | real={class_names[int(y_true[idx])]} | pred={class_names[int(y_pred[idx])]} | p={prob:.3f}",
            fontsize=10,
        )
        plt.axis("off")
    plt.tight_layout()
    path = os.path.join(EXAMPLES_DIR, f"{run_id}_correct_incorrect_examples.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    return path


error_rows = []
for model_id, payload in BEST_RUN_CACHE.items():
    row = payload["row"]
    y_true = payload["y_true"]
    y_pred = payload["y_pred"]
    cm = confusion_matrix(y_true, y_pred)
    np.fill_diagonal(cm, 0)
    max_pair = np.unravel_index(np.argmax(cm), cm.shape)
    error_rows.append({
        "model_id": model_id,
        "display_name": row["display_name"],
        "top_confusion_real": class_names[max_pair[0]],
        "top_confusion_pred": class_names[max_pair[1]],
        "count": int(cm[max_pair]),
    })
    if payload["family"] == "keras":
        save_correct_incorrect_examples(payload["x_test"], y_true, y_pred, payload["y_prob"], f"{model_id}_best")

error_df = pd.DataFrame(error_rows)
error_df.to_csv(os.path.join(METRICS_DIR, "error_analysis.csv"), index=False)
display(error_df)

## 10. Grad-CAM

Aplicamos Grad-CAM al mejor modelo justo para visualizar qué regiones de la imagen influyen más en la predicción.


In [ ]:
def overlay_heatmap(img, heatmap, alpha=0.35):
    img_float = img.astype("float32")
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = heatmap_color * alpha + img_float * (1 - alpha)
    return np.clip(overlay, 0, 255).astype("uint8")


def find_last_conv_layer_keras(model):
    priority_names = ["last_conv_custom", "gradcam_resnet50_output"]
    top_names = [layer.name for layer in model.layers]
    for name in priority_names:
        if name in top_names:
            return model.get_layer(name)
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer
        if isinstance(layer, keras.Model):
            # Fallback informativo: las capas internas pueden dar grafo desconectado si no se expone su salida.
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, layers.Conv2D):
                    return sublayer
    raise ValueError("No se encontró una capa compatible para Grad-CAM en el modelo Keras.")


def gradcam_keras(model, img_array, layer_name=None):
    if layer_name is None:
        target_layer = find_last_conv_layer_keras(model)
        layer_name = target_layer.name
    else:
        target_layer = model.get_layer(layer_name)
    grad_model = keras.Model(model.inputs, [target_layer.output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


class TorchGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.fwd = target_layer.register_forward_hook(self._forward_hook)
        self.bwd = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, target_class=None):
        self.model.eval()
        x = x.to(device)
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        probs = torch.softmax(logits, dim=1)
        pred = int(torch.argmax(probs, dim=1).item())
        class_idx = pred if target_class is None else int(target_class)
        loss = logits[:, class_idx].sum()
        loss.backward()
        weights = self.gradients[0].mean(dim=(1, 2), keepdim=True)
        cam = (weights * self.activations[0]).sum(dim=0)
        cam = F.relu(cam).detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam, probs.detach().cpu().numpy()[0], pred

    def close(self):
        self.fwd.remove()
        self.bwd.remove()


def find_last_conv_layer_torch(model):
    conv_layers = [module for module in model.modules() if isinstance(module, nn.Conv2d)]
    if not conv_layers:
        raise ValueError("No se encontró una capa convolucional en el modelo PyTorch.")
    return conv_layers[-1]


def select_best_model_ids():
    fair_ids = [mid for mid in summary_df["model_id"].tolist() if mid != "M6_MAXNET_ISIC_DERMATOLOGY_FROZEN_BASELINE"]
    fair_df = summary_df[summary_df["model_id"].isin(fair_ids)].sort_values("f1_macro_mean", ascending=False)
    best_fair = fair_df.iloc[0]["model_id"] if not fair_df.empty else None
    best_overall = summary_df.sort_values("f1_macro_mean", ascending=False).iloc[0]["model_id"] if not summary_df.empty else None
    return best_fair, best_overall


def collect_gradcam_indices(y_true, y_pred, max_correct=3, max_incorrect=3):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    correct_idx = np.where(y_true == y_pred)[0].tolist()
    incorrect_idx = np.where(y_true != y_pred)[0].tolist()
    per_class = {}
    for class_idx in range(NUM_CLASSES):
        matches = [idx for idx in correct_idx if int(y_true[idx]) == class_idx]
        if matches:
            per_class[class_idx] = matches[0]
    return {
        "correct": correct_idx[:max_correct],
        "incorrect": incorrect_idx[:max_incorrect],
        "per_class": per_class,
    }


def denormalize_torch_image(tensor, mode):
    arr = tensor.detach().cpu().permute(1, 2, 0).numpy()
    if mode == "minus_one_one":
        arr = arr * 0.5 + 0.5
    elif mode == "imagenet":
        mean = np.array([0.485, 0.456, 0.406], dtype="float32")
        std = np.array([0.229, 0.224, 0.225], dtype="float32")
        arr = arr * std + mean
    arr = np.clip(arr, 0, 1)
    return (arr * 255).astype("uint8")


def save_gradcam_figure(raw_img, heatmap, title, output_path):
    overlay = overlay_heatmap(raw_img, heatmap)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_img)
    axes[0].set_title("Original")
    axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title("Heatmap")
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def render_gradcam_examples_for_keras(payload, model_id):
    model = payload["model"]
    x_test = payload["x_test"]
    groups = collect_gradcam_indices(payload["y_true"], payload["y_pred"])
    base_dir = os.path.join(GRADCAM_DIR, model_id)
    for folder in ("correct", "incorrect", "per_class"):
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    for rank, idx in enumerate(groups["correct"], start=1):
        raw = x_test[idx]
        heatmap = gradcam_keras(model, np.expand_dims(raw, axis=0))
        prob = float(np.max(payload["y_prob"][idx]))
        title = f"{model_id} | correct #{rank} | idx={idx} | real={class_names[int(payload['y_true'][idx])]} | pred={class_names[int(payload['y_pred'][idx])]} | p={prob:.3f}"
        save_gradcam_figure(raw, heatmap, title, os.path.join(base_dir, "correct", f"correct_{rank:02d}_idx{idx}.png"))

    for rank, idx in enumerate(groups["incorrect"], start=1):
        raw = x_test[idx]
        heatmap = gradcam_keras(model, np.expand_dims(raw, axis=0))
        prob = float(np.max(payload["y_prob"][idx]))
        title = f"{model_id} | incorrect #{rank} | idx={idx} | real={class_names[int(payload['y_true'][idx])]} | pred={class_names[int(payload['y_pred'][idx])]} | p={prob:.3f}"
        save_gradcam_figure(raw, heatmap, title, os.path.join(base_dir, "incorrect", f"incorrect_{rank:02d}_idx{idx}.png"))

    for class_idx, idx in groups["per_class"].items():
        raw = x_test[idx]
        heatmap = gradcam_keras(model, np.expand_dims(raw, axis=0))
        prob = float(np.max(payload["y_prob"][idx]))
        title = f"{model_id} | per_class | class={class_names[class_idx]} | idx={idx} | pred={class_names[int(payload['y_pred'][idx])]} | p={prob:.3f}"
        save_gradcam_figure(raw, heatmap, title, os.path.join(base_dir, "per_class", f"class_{class_idx:02d}_idx{idx}.png"))


def render_gradcam_examples_for_torch(payload, model_id):
    model = payload["model"]
    test_dataset = payload["test_loader"].dataset
    groups = collect_gradcam_indices(payload["y_true"], payload["y_pred"])
    base_dir = os.path.join(GRADCAM_DIR, model_id)
    for folder in ("correct", "incorrect", "per_class"):
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    cam = TorchGradCAM(model, find_last_conv_layer_torch(model))
    try:
        for rank, idx in enumerate(groups["correct"], start=1):
            tensor, _ = test_dataset[idx]
            heatmap, _, _ = cam(tensor.unsqueeze(0), target_class=int(payload["y_pred"][idx]))
            raw = denormalize_torch_image(tensor, payload.get("denorm", "minus_one_one"))
            prob = float(np.max(payload["y_prob"][idx]))
            title = f"{model_id} | correct #{rank} | idx={idx} | real={class_names[int(payload['y_true'][idx])]} | pred={class_names[int(payload['y_pred'][idx])]} | p={prob:.3f}"
            save_gradcam_figure(raw, heatmap, title, os.path.join(base_dir, "correct", f"correct_{rank:02d}_idx{idx}.png"))

        for rank, idx in enumerate(groups["incorrect"], start=1):
            tensor, _ = test_dataset[idx]
            heatmap, _, _ = cam(tensor.unsqueeze(0), target_class=int(payload["y_pred"][idx]))
            raw = denormalize_torch_image(tensor, payload.get("denorm", "minus_one_one"))
            prob = float(np.max(payload["y_prob"][idx]))
            title = f"{model_id} | incorrect #{rank} | idx={idx} | real={class_names[int(payload['y_true'][idx])]} | pred={class_names[int(payload['y_pred'][idx])]} | p={prob:.3f}"
            save_gradcam_figure(raw, heatmap, title, os.path.join(base_dir, "incorrect", f"incorrect_{rank:02d}_idx{idx}.png"))

        for class_idx, idx in groups["per_class"].items():
            tensor, _ = test_dataset[idx]
            heatmap, _, _ = cam(tensor.unsqueeze(0), target_class=int(payload["y_pred"][idx]))
            raw = denormalize_torch_image(tensor, payload.get("denorm", "minus_one_one"))
            prob = float(np.max(payload["y_prob"][idx]))
            title = f"{model_id} | per_class | class={class_names[class_idx]} | idx={idx} | pred={class_names[int(payload['y_pred'][idx])]} | p={prob:.3f}"
            save_gradcam_figure(raw, heatmap, title, os.path.join(base_dir, "per_class", f"class_{class_idx:02d}_idx{idx}.png"))
    finally:
        cam.close()


BEST_FAIR_MODEL, BEST_OVERALL_MODEL = select_best_model_ids()
print("BEST_FAIR_MODEL:", BEST_FAIR_MODEL)
print("BEST_OVERALL_MODEL:", BEST_OVERALL_MODEL)

if BEST_FAIR_MODEL in BEST_RUN_CACHE:
    payload = BEST_RUN_CACHE[BEST_FAIR_MODEL]
    if payload["family"] == "keras":
        render_gradcam_examples_for_keras(payload, BEST_FAIR_MODEL)
    elif payload["family"] == "torch":
        render_gradcam_examples_for_torch(payload, BEST_FAIR_MODEL)
    print("Grad-CAM generado para el mejor modelo justo.")
else:
    print("No hay ejecuciones válidas en BEST_RUN_CACHE para Grad-CAM.")


### Informes y archivos de salida


In [ ]:
def write_markdown(path: str, text: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


fairness_note = (
    "Para garantizar una comparación equilibrada, los modelos shallow y deep parcial "
    "se entrenaron con el mismo número máximo de épocas. En ambos casos se usó "
    "EarlyStopping monitorizando val_macro_f1, por lo que el número efectivo de épocas "
    "puede variar entre modelos, pero el presupuesto máximo de entrenamiento fue equivalente."
)

isic_note = (
    "DermaMNIST deriva de HAM10000. Por ello, el modelo EfficientNet-B1 preentrenado "
    "en el entorno ISIC se mantiene congelado y solo se entrena la cabeza de clasificación. "
    "Sus resultados se interpretan como una referencia dermatológica de dominio cercano, "
    "no como una comparación completamente independiente."
)


audit_sections = [
    "# Revisión previa",
    "",
    AUDIT_INFO["summary"],
    "",
    "## Elementos reutilizados",
    *[f"- {item}" for item in AUDIT_INFO["reused"]],
    "",
    "## Elementos descartados",
    *[f"- {item}" for item in AUDIT_INFO["discarded"]],
]
write_markdown(os.path.join(REPORTS_DIR, "file_audit.md"), "\n".join(audit_sections) + "\n")


methodology_sections = [
    "# Methodology Notes",
    "",
    "- El trabajo se centra en clasificación multiclase de imágenes con DermaMNIST.",
    f"- {isic_note}",
    "- La métrica principal de comparación es `f1_macro`, seguida de `balanced_accuracy`.",
    "- Shallow tuning: máximo 30 épocas por seed.",
    "- Deep tuning parcial: máximo 30 épocas por seed.",
    "- EarlyStopping monitorizando `val_macro_f1`.",
    "- La selección del mejor checkpoint se hace por `val_macro_f1`.",
    "- La evaluación final en test se realiza una sola vez por seed, tras cargar el mejor checkpoint de validación.",
    "- Se utiliza la misma estrategia de pesos de clase en los seis modelos principales.",
    "",
    "## Relación con el temario",
    "",
    "- El trabajo se centra en clasificación de imágenes, porque DermaMNIST proporciona etiquetas por imagen.",
    "- La CNN propia usa bloques Conv2D, BatchNorm, ReLU, MaxPooling, Dropout y softmax.",
    "- ResNet50 ImageNet permite estudiar transferencia generalista.",
    "- RadImageNet/RAC permite estudiar transferencia desde imagen médica no dermatológica.",
    "- EfficientNet-B1 ISIC se usa como baseline dermatológico congelado.",
    "- Grad-CAM se emplea para interpretar visualmente el modelo.",
    "",
    f"> {fairness_note}",
]
write_markdown(os.path.join(REPORTS_DIR, "methodology_notes.md"), "\n".join(methodology_sections) + "\n")


ordered_summary = summary_df.copy()
ordered_summary["model_order"] = ordered_summary["model_id"].map(lambda x: MODEL_ORDER.index(x) if x in MODEL_ORDER else 999)
ordered_summary = ordered_summary.sort_values(["model_order", "display_name"]).reset_index(drop=True)

top_lines = []
for _, row in ordered_summary.iterrows():
    top_lines.append(
        f"- {row['display_name']}: F1_macro={row['f1_macro_mean_pm_std']}, "
        f"BalancedAccuracy={row['balanced_accuracy_mean_pm_std']}, "
        f"Kappa={row['cohen_kappa_mean_pm_std']}"
    )

def model_name_from_id(model_id):
    if model_id is None:
        return "No disponible"
    return MODEL_DISPLAY_NAMES.get(model_id, model_id)


final_sections = [
    "# Final Summary",
    "",
    "## 1. Objetivo",
    "Comparar entrenamiento desde cero, transferencia generalista, transferencia médica no dermatológica y una referencia dermatológica de dominio cercano sobre DermaMNIST.",
    "",
    "## 2. Dataset y particiones",
    f"Se utiliza DermaMNIST con splits oficiales `train/val/test`, resolución {IMG_SIZE}x{IMG_SIZE} y formato RGB.",
    "",
    "## 3. Modelos comparados",
    "- M1: CNN propia",
    "- M2: ResNet50 ImageNet shallow",
    "- M3: ResNet50 ImageNet deep",
    "- M4: ResNet50 RadImageNet/RAC shallow",
    "- M5: ResNet50 RadImageNet/RAC deep",
    "- M6: EfficientNet-B1 ISIC frozen",
    "",
    "## 4. Diseño experimental",
    "- Mismo máximo de 30 épocas para shallow y deep parcial.",
    "- EarlyStopping y checkpoint monitorizando `val_macro_f1`.",
    "- Evaluación final en test una sola vez por seed.",
    "- Misma estrategia de pesos de clase para todos los modelos.",
    "",
    fairness_note,
    "",
    "## 5. Resultados mean ± std",
    *(top_lines if top_lines else ["- No hay resultados agregados todavía."]),
    "",
    "## 6. Mejor modelo justo",
    f"- {model_name_from_id(BEST_FAIR_MODEL)}",
    "",
    "## 7. Referencia dermatológica",
    f"- {isic_note}",
    "",
    "## 8. Desbalanceo",
    "Se priorizan Macro F1 y Balanced accuracy por el desbalanceo entre clases.",
    "",
    "## 9. Grad-CAM",
    "Se generan ejemplos correctos, incorrectos y por clase para el mejor modelo justo, siempre que exista una ejecución válida.",
    "",
    "## 10. Conclusiones",
    "- La comparación mantiene un criterio homogéneo entre familias de modelos.",
    "- La transferencia generalista y médica puede compararse de forma directa con el mismo presupuesto máximo de entrenamiento.",
    "- El modelo ISIC se interpreta como referencia de dominio cercano y no como comparación completamente independiente.",
    "- Todos los resultados principales quedan guardados en Drive.",
]
write_markdown(os.path.join(REPORTS_DIR, "final_summary.md"), "\n".join(final_sections) + "\n")

print("Informes generados en:", REPORTS_DIR)


## 11. Conclusiones

En este trabajo comparamos distintas formas de transferencia de conocimiento sobre DermaMNIST con un criterio homogéneo de entrenamiento y evaluación. El modelo EfficientNet-B1 preentrenado en ISIC se interpreta como una referencia dermatológica de dominio cercano, no como una comparación completamente independiente.


In [ ]:
# ============================================================
# Tabla final con media ± desviación típica por bootstrap en TEST
# ============================================================

import os
import glob
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score,
    cohen_kappa_score,
)

BOOTSTRAP_B = 1000
BOOTSTRAP_SEED = 123
USE_STRATIFIED_BOOTSTRAP = True


def compute_test_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
        ),
    }


def bootstrap_metric_std_test(
    y_true,
    y_pred,
    n_bootstrap=1000,
    seed=123,
    stratified=True,
):
    """
    Estima media y desviación típica de métricas en test mediante bootstrap.
    Si stratified=True, remuestrea dentro de cada clase para mantener la distribución
    de clases del test original.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    n = len(y_true)
    boot_rows = []

    if stratified:
        class_indices = {
            c: np.where(y_true == c)[0]
            for c in np.arange(NUM_CLASSES)
        }

    for _ in range(n_bootstrap):
        if stratified:
            sampled_parts = []
            for c, idx_c in class_indices.items():
                if len(idx_c) == 0:
                    continue
                sampled_c = rng.choice(idx_c, size=len(idx_c), replace=True)
                sampled_parts.append(sampled_c)
            idx = np.concatenate(sampled_parts)
            rng.shuffle(idx)
        else:
            idx = rng.choice(np.arange(n), size=n, replace=True)

        metrics = compute_test_metrics(y_true[idx], y_pred[idx])
        boot_rows.append(metrics)

    boot_df = pd.DataFrame(boot_rows)

    summary = {}

    for col in boot_df.columns:
        summary[f"{col}_boot_mean"] = float(boot_df[col].mean())
        summary[f"{col}_boot_std"] = float(boot_df[col].std(ddof=1))
        summary[f"{col}_boot_p025"] = float(boot_df[col].quantile(0.025))
        summary[f"{col}_boot_p975"] = float(boot_df[col].quantile(0.975))

    return summary, boot_df


def infer_model_id_from_predictions_filename(path):
    name = os.path.basename(path)
    name = name.replace("_predictions.csv", "")

    # Ejemplo:
    # M1_CNN_PROPIA_224_seed123 -> model_id=M1_CNN_PROPIA_224, seed=123
    if "_seed" in name:
        model_id, seed_part = name.rsplit("_seed", 1)
        try:
            seed = int(seed_part)
        except Exception:
            seed = None
    else:
        model_id = name
        seed = None

    return model_id, seed


def get_display_name(model_id):
    if "MODEL_DISPLAY_NAMES" in globals() and model_id in MODEL_DISPLAY_NAMES:
        return MODEL_DISPLAY_NAMES[model_id]
    return model_id


prediction_paths = sorted(glob.glob(os.path.join(PREDICTIONS_DIR, "*_predictions.csv")))

if not prediction_paths:
    raise FileNotFoundError(f"No se han encontrado predicciones en: {PREDICTIONS_DIR}")

bootstrap_summary_rows = []
bootstrap_all_dir = os.path.join(SUMMARY_DIR, "bootstrap_test")
os.makedirs(bootstrap_all_dir, exist_ok=True)

for pred_path in prediction_paths:
    model_id, seed = infer_model_id_from_predictions_filename(pred_path)

    df_pred = pd.read_csv(pred_path)

    # Intento robusto de nombres de columnas
    possible_true_cols = ["y_true", "true_label", "label", "true"]
    possible_pred_cols = ["y_pred", "pred_label", "prediction", "pred"]

    y_true_col = next((c for c in possible_true_cols if c in df_pred.columns), None)
    y_pred_col = next((c for c in possible_pred_cols if c in df_pred.columns), None)

    if y_true_col is None or y_pred_col is None:
        print("Saltando archivo porque no encuentro y_true/y_pred:", pred_path)
        print("Columnas:", df_pred.columns.tolist())
        continue

    y_true_i = df_pred[y_true_col].values.astype(int)
    y_pred_i = df_pred[y_pred_col].values.astype(int)

    base_metrics = compute_test_metrics(y_true_i, y_pred_i)
    boot_summary, boot_df = bootstrap_metric_std_test(
        y_true_i,
        y_pred_i,
        n_bootstrap=BOOTSTRAP_B,
        seed=BOOTSTRAP_SEED,
        stratified=USE_STRATIFIED_BOOTSTRAP,
    )

    row = {
        "model_id": model_id,
        "display_name": get_display_name(model_id),
        "seed": seed,
        "n_test": int(len(y_true_i)),
        "bootstrap_B": BOOTSTRAP_B,
        "bootstrap_stratified": USE_STRATIFIED_BOOTSTRAP,
    }

    row.update({f"{k}_test": float(v) for k, v in base_metrics.items()})
    row.update(boot_summary)

    bootstrap_summary_rows.append(row)

    boot_out_path = os.path.join(
        bootstrap_all_dir,
        f"{model_id}_seed{seed}_bootstrap_test_metrics.csv"
    )
    boot_df.to_csv(boot_out_path, index=False)

bootstrap_results_df = pd.DataFrame(bootstrap_summary_rows)

# Ordenar según MODEL_ORDER si existe
if "MODEL_ORDER" in globals():
    bootstrap_results_df["model_order"] = bootstrap_results_df["model_id"].map(
        lambda x: MODEL_ORDER.index(x) if x in MODEL_ORDER else 999
    )
    bootstrap_results_df = bootstrap_results_df.sort_values(["model_order", "seed"])
else:
    bootstrap_results_df = bootstrap_results_df.sort_values(["model_id", "seed"])


def pm_std(row, metric, decimals=3):
    return f"{row[f'{metric}_boot_mean']:.{decimals}f} ± {row[f'{metric}_boot_std']:.{decimals}f}"


table_bootstrap_test = pd.DataFrame({
    "Modelo": bootstrap_results_df["display_name"],
    "Acc.": bootstrap_results_df.apply(lambda r: pm_std(r, "accuracy"), axis=1),
    "Bal. Acc.": bootstrap_results_df.apply(lambda r: pm_std(r, "balanced_accuracy"), axis=1),
    "Kappa": bootstrap_results_df.apply(lambda r: pm_std(r, "cohen_kappa"), axis=1),
    "Macro Rec.": bootstrap_results_df.apply(lambda r: pm_std(r, "recall_macro"), axis=1),
    "Macro F1": bootstrap_results_df.apply(lambda r: pm_std(r, "f1_macro"), axis=1),
    "n test": bootstrap_results_df["n_test"].astype(int),
})

display(table_bootstrap_test)

bootstrap_results_df.to_csv(
    os.path.join(SUMMARY_DIR, "bootstrap_test_summary_full.csv"),
    index=False,
)

table_bootstrap_test.to_csv(
    os.path.join(SUMMARY_DIR, "tabla_final_test_bootstrap_mean_std.csv"),
    index=False,
)

table_bootstrap_test.to_excel(
    os.path.join(SUMMARY_DIR, "tabla_final_test_bootstrap_mean_std.xlsx"),
    index=False,
)

print("Guardado:")
print(os.path.join(SUMMARY_DIR, "tabla_final_test_bootstrap_mean_std.csv"))
print(os.path.join(SUMMARY_DIR, "tabla_final_test_bootstrap_mean_std.xlsx"))
print(os.path.join(SUMMARY_DIR, "bootstrap_test_summary_full.csv"))

In [ ]:
print("Estado de ejecución por modelo:")
for model_id, status in MODEL_STATUSES.items():
    label = MODEL_DISPLAY_NAMES.get(model_id, model_id)
    print(f"- {label}: {status}")

print("\nArchivos principales:")
print("- all_results_long.csv ->", os.path.join(SUMMARY_DIR, "all_results_long.csv"))
print("- summary_mean_std.csv ->", os.path.join(SUMMARY_DIR, "summary_mean_std.csv"))
print("- file_audit.md ->", os.path.join(REPORTS_DIR, "file_audit.md"))
print("- final_summary.md ->", os.path.join(REPORTS_DIR, "final_summary.md"))
print("- methodology_notes.md ->", os.path.join(REPORTS_DIR, "methodology_notes.md"))
print("- gradcam dir ->", GRADCAM_DIR)
print("- logs/run_errors.txt ->", os.path.join(LOG_DIR, "run_errors.txt"))
